# CF-TTA: Counterfactual Test-Time Adaptation with Label-Free Adaptive Decision Correction for Cross-Domain Diabetic Retinopathy Grading

## balanced of 708 per class training seed 2


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split

# =====================================================
# LOAD EYEPACS LABELS
# =====================================================
labels_path = (
    "/kaggle/input/datasets/dreamer07/"
    "eyepacs/trainLabels.csv/trainLabels.csv"
)

df = pd.read_csv(labels_path)

# =====================================================
# STANDARDIZE COLUMN NAMES
# =====================================================
df = df.rename(
    columns={
        "image": "id_code",
        "level": "diagnosis"
    }
)

print("\n📊 Original Distribution")
print(df["diagnosis"].value_counts())

# =====================================================
# CREATE BALANCED 708 / CLASS DATASET
# =====================================================
balanced = []

for cls in range(5):

    cls_df = df[df["diagnosis"] == cls]

    sampled = cls_df.sample(
        n=708,
        random_state=42
    )

    balanced.append(sampled)

balanced_df = pd.concat(balanced)

balanced_df = balanced_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\n📊 Balanced Distribution")
print(balanced_df["diagnosis"].value_counts())

# =====================================================
# SAVE FULL BALANCED CSV
# =====================================================
balanced_csv = (
    "/kaggle/working/"
    "balanced_708_per_class.csv"
)

balanced_df.to_csv(
    balanced_csv,
    index=False
)

print(f"\n✅ Saved: {balanced_csv}")

# =====================================================
# CREATE TRAIN / VAL / TEST SPLITS
#
# 70 / 15 / 15
# stratified
# =====================================================

train_df, temp_df = train_test_split(

    balanced_df,

    test_size=0.30,

    stratify=balanced_df["diagnosis"],

    random_state=42
)

val_df, test_df = train_test_split(

    temp_df,

    test_size=0.50,

    stratify=temp_df["diagnosis"],

    random_state=42
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

split_df = pd.concat([
    train_df,
    val_df,
    test_df
])

split_df = split_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\n📊 Split Distribution")

print(
    split_df.groupby(
        ["split", "diagnosis"]
    ).size()
)

# =====================================================
# SAVE SPLIT CSV
# =====================================================
split_csv = (
    "/kaggle/working/"
    "split.csv"
)

split_df.to_csv(
    split_csv,
    index=False
)

print(f"\n✅ Saved: {split_csv}")


📊 Original Distribution
diagnosis
0    25810
2     5292
1     2443
3      873
4      708
Name: count, dtype: int64

📊 Balanced Distribution
diagnosis
1    708
0    708
2    708
4    708
3    708
Name: count, dtype: int64

✅ Saved: /kaggle/working/balanced_708_per_class.csv

📊 Split Distribution
split  diagnosis
test   0            107
       1            106
       2            106
       3            106
       4            106
train  0            495
       1            496
       2            496
       3            495
       4            496
val    0            106
       1            106
       2            106
       3            107
       4            106
dtype: int64

✅ Saved: /kaggle/working/split.csv


In [22]:
%%writefile /kaggle/working/config.py

import torch

class CFG:

    # =====================================================
    # EYEPACS TRAINING DATA
    # =====================================================
    labels_path = (
        "/kaggle/input/datasets/dreamer07/"
        "eyepacs/trainLabels.csv/trainLabels.csv"
    )

    img_dir = (
        "/kaggle/input/datasets/dreamer07/"
        "eyepacs/data/data"
    )

    # =====================================================
    # FUNOTTA FAIR SUBSET  (708 per class, balanced)
    # =====================================================
    subset_csv = (
        "/kaggle/working/balanced_708_per_class.csv"
    )

    split_csv = (
        "/kaggle/working/split.csv"
    )

    # =====================================================
    # IMAGE
    # =====================================================
    img_size = 320

    # =====================================================
    # BATCH
    # =====================================================
    batch_size = 16

    # =====================================================
    # DATALOADER
    # =====================================================
    num_workers = 2

    pin_memory = True

    persistent_workers = True

    prefetch_factor = 2

    # =====================================================
    # MODEL
    # =====================================================
    num_classes = 5

    # =====================================================
    # TRAINING
    # =====================================================
    epochs = 40

    lr = 1.5e-4

    weight_decay = 8e-5

    ordinal_weight = 0.60

    label_smoothing = 0.05

    use_amp = True

    grad_clip = 1.0

    # =====================================================
    # DEVICE
    # =====================================================
    device = torch.device(

        "cuda"

        if torch.cuda.is_available()

        else "cpu"
    )

    # =====================================================
    # MULTI GPU
    # =====================================================
    use_multi_gpu = (

        torch.cuda.device_count() > 1
    )

    # =====================================================
    # CUDA
    # =====================================================
    cudnn_benchmark = False

    seed = 2026
    save_path = "/kaggle/input/models/knightcracker/seed2026-model/pytorch/default/1/best_dsr50_balanced_seed_2026.pth"

    # =========================================================
    # =========================================================
    # CF-NODE TTA SETTINGS  (IEEE Access final, v2)
    # =========================================================
    # =========================================================

    # =====================================================
    # TARGET DATASET
    #
    # IMPORTANT:
    # overwritten dynamically in run.py.
    # run.py should set BOTH:
    #     cfg.target_dataset = name
    #     cfg.target_name    = name
    # so per-target cf_strength resolves correctly.
    # =====================================================
    target_dataset = "messidor"

    target_name    = "messidor"

    # =====================================================
    # ENABLES
    # =====================================================
    use_prior_correction = True

    use_cf_adaptation = True

    skip_cf_for_deepdrid = True

    use_aggressive_cf = False

    # =====================================================
    # SOURCE PRIOR
    #
    # UNIFORM — matches the 708-per-class balanced
    # training distribution.  An EyePACS-skewed prior
    # on a uniformly-trained model collapses QWK on
    # APTOS / MESSIDOR.  See paper Sec. IV-C.
    # =====================================================
    source_prior = [

        0.20,

        0.20,

        0.20,

        0.20,

        0.20,
    ]

    # =====================================================
    # PRIOR CORRECTION STRENGTH
    #
    # No-op under a uniform prior (additive constant in
    # log-space), retained for cfg parity with the full-
    # EyePACS configuration.
    # =====================================================
    prior_correction_strength = 0.03

    # =====================================================
    # BAYES-OPTIMAL QWK PREDICTOR  (Proposition 1)
    #
    # "argmax"   -- conventional MAP estimator
    # "expected" -- round of posterior expected grade
    # "hybrid"   -- argmax, except when |argmax - round(E[k])|
    #               >= 2 (then use round(E[k])).  DEFAULT.
    # =====================================================
    predict_mode = "hybrid"

    # =====================================================
    # PER-TARGET CF STRENGTH
    #
    # Global multiplier on the causal-gated ordinal blend
    # weight alpha.  Resolved at run-time against
    # cfg.target_name.
    #
    # 'default' is used if the target name is not in the
    # dict (defensive fallback).
    # =====================================================
        # =====================================================
    # PER-TARGET CF STRENGTH
    #
    # All 1.0 reproduces the v1 algorithm exactly.
    # =====================================================
    cf_strength = {

        "messidor": 1.00,

        "idrid":    1.00,

        "ddr":      1.00,

        "aptos":    1.00,

        "deepdrid": 1.00,

        "default":  1.00,
    }

    # SOURCE PRIOR — EyePACS empirical (v1, the working setting)
   # source_prior = [
#
 #       0.73,
##       0.07,
#
 #       0.15,
#
 #       0.025,
#
 #       0.025,
  #  ]
#
    source_prior = [
        0.734783,
        0.069550,
        0.150658,
        0.024853,
        0.020156,
    ]

    # PRIOR CORRECTION STRENGTH (v1 default)
    prior_correction_strength = 0.18

    # =====================================================
    # SENSITIVITY  (causal gate threshold)
    # =====================================================
    sensitivity_threshold = 0.12

    # =====================================================
    # CONFIDENCE THRESHOLD  (legacy, retained)
    # =====================================================
    high_conf = 0.84

    # =====================================================
    # TEMPERATURE  (base softmax temperature)
    # =====================================================
    temp = 1.05

    # =====================================================
    # COUNTERFACTUAL SUPPRESSION SCHEDULE
    #
    # Set to None to use the default 4-scale schedule:
    #   ((0.02, 0.75), (0.05, 0.40),
    #    (0.10, 0.20), (0.15, 0.10))
    # =====================================================
    cf_levels = None

    # =====================================================
    # LEGACY  (retained for backward compatibility)
    # =====================================================
    prior_strength = 0.03

    max_cf_alpha   = 0.015

    tta_steps      = 1

    tta_lr         = 2e-5

    margin         = 0.12

    lambda_tta     = 0.35

    noise_weight   = 0.15

    w_floor        = 0.55

    aug_weight     = 0.20

    entropy_weight = 0.003

    use_blur       = False

    use_entropy    = False

    # =========================================================
    # =========================================================
    # UNIFIED TEST DATASETS
    # =========================================================
    # =========================================================

    # -----------------------------------------------------
    # IDRID
    # -----------------------------------------------------
    idrid_img_dir = (
        "/kaggle/input/dr-datasets/IDRiD/Images"
    )

    idrid_csv = (
        "/kaggle/input/dr-datasets/IDRiD/idrid_labels.csv"
    )

    # -----------------------------------------------------
    # DDR
    # -----------------------------------------------------
    ddr_img_dir = (
        "/kaggle/input/dr-datasets/DDR"
    )

    ddr_csv = (
        "/kaggle/input/dr-datasets/DDR/DR_grading.csv"
    )

    # -----------------------------------------------------
    # MESSIDOR
    # -----------------------------------------------------
    messidor_img_dir = (
        "/kaggle/input/dr-datasets/MESSIDOR"
    )

    messidor_csv = (
        "/kaggle/input/dr-datasets/MESSIDOR/messidor_labels.csv"
    )

    # -----------------------------------------------------
    # DEEPDRID
    # -----------------------------------------------------
    deepdrid_img_dir = (
        "/kaggle/input/dr-datasets/DeepDRiD"
    )

    deepdrid_csv = (
        "/kaggle/input/dr-datasets/DeepDRiD/deepdrid_labels.csv"
    )

    # -----------------------------------------------------
    # APTOS
    # -----------------------------------------------------
    aptos_img_dir = (
        "/kaggle/input/dr-datasets/APTOS"
    )

    aptos_csv = (
        "/kaggle/input/dr-datasets/APTOS/train.csv"
    )


        # =====================================================
    # LOGIT-ADJUSTED PREDICTION  (Menon et al., ICLR 2021)
    #
    # Single global scalar — applied uniformly to every
    # target.  Uses only the source prior; no target-domain
    # statistics.  Affects argmax only — AUC ranking score
    # is unchanged.
    #
    # 0.0 -> v1 argmax (no boost)
    # 0.5 -> recommended starting value
    # 1.0 -> Menon default (full prior unwinding)
    # =====================================================
    pred_logit_adjust = 0.30

    self.prior_correction_strength = 0.18

Overwriting /kaggle/working/config.py


In [9]:
%%writefile /kaggle/working/dataset.py

import os

import random

import numpy as np

import pandas as pd

from PIL import Image

import torch

from torch.utils.data import (
    Dataset,
    DataLoader
)

import torchvision.transforms as transforms


# =========================================================
# FINAL FUNOTTA-FAIR DATASET PIPELINE
#
# PURPOSE:
# - exact fair comparison vs FunOTTA
# - use official 708/class balanced subset
# - preserve strong F1
# - preserve AUC
# - reduce overfitting on 3540 images
# - improve cross-domain generalization
# =========================================================


# =========================================================
# TRANSFORMS
# =========================================================
def get_transforms(
    img_size,
    mode="train"
):

    if mode == "train":

        return transforms.Compose([

            # =============================================
            # SMALL-DATA GENERALIZATION
            # =============================================
            transforms.RandomResizedCrop(

                img_size,

                scale=(0.88, 1.0),

                ratio=(0.95, 1.05)
            ),

            # =============================================
            # SAFE RETINAL AUGMENTATIONS
            # =============================================
            transforms.RandomHorizontalFlip(
                p=0.5
            ),

            transforms.RandomRotation(
                degrees=8
            ),

            # =============================================
            # COLOR AUGMENTATION
            # =============================================
            transforms.ColorJitter(

                brightness=0.08,

                contrast=0.08,

                saturation=0.05,

                hue=0.01
            ),

            # =============================================
            # LIGHT BLUR
            # =============================================
            transforms.RandomApply(

                [
                    transforms.GaussianBlur(
                        kernel_size=3,
                        sigma=(0.1, 1.0)
                    )
                ],

                p=0.15
            ),

            # =============================================
            # TENSOR
            # =============================================
            transforms.ToTensor(),

            # =============================================
            # NORMALIZATION
            # =============================================
            transforms.Normalize(

                [0.485, 0.456, 0.406],

                [0.229, 0.224, 0.225]
            )
        ])

    else:

        return transforms.Compose([

            transforms.Resize(
                (img_size, img_size)
            ),

            transforms.ToTensor(),

            transforms.Normalize(

                [0.485, 0.456, 0.406],

                [0.229, 0.224, 0.225]
            )
        ])


# =========================================================
# DATASET
# =========================================================
class EyePACSDataset(Dataset):

    def __init__(
        self,
        df,
        img_dir,
        transform=None
    ):

        self.df = df.reset_index(
            drop=True
        )

        self.img_dir = img_dir

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img_id = str(
            row["id_code"]
        )

        label = int(
            row["diagnosis"]
        )

        path = os.path.join(

            self.img_dir,

            img_id + ".jpeg"
        )

        try:

            image = Image.open(
                path
            ).convert("RGB")

        except Exception:

            print(
                f"⚠️ Failed loading: {path}"
            )

            image = Image.new(
                "RGB",
                (320, 320)
            )

        if self.transform:

            image = self.transform(
                image
            )

        return (

            image,

            torch.tensor(
                label,
                dtype=torch.long
            )
        )


# =========================================================
# WORKER SEEDING
# =========================================================
def seed_worker(worker_id):

    worker_seed = (
        torch.initial_seed()
        % 2**32
    )

    np.random.seed(
        worker_seed
    )

    random.seed(
        worker_seed
    )


# =========================================================
# LOADERS
# =========================================================
def get_loaders(cfg):

    # =====================================================
    # LOAD BALANCED SUBSET
    # =====================================================
    subset_df = pd.read_csv(
        cfg.subset_csv
    )

    # =====================================================
    # LOAD SPLITS
    # =====================================================
    split_df = pd.read_csv(
        cfg.split_csv
    )

    # =====================================================
    # STANDARDIZE COLUMN NAMES
    # =====================================================
    subset_df = subset_df.rename(
        columns={
            "image": "id_code",
            "level": "diagnosis"
        }
    )

    split_df = split_df.rename(
        columns={
            "image": "id_code",
            "level": "diagnosis"
        }
    )

    # =====================================================
    # VERIFY COLUMNS
    # =====================================================
    print("\nSubset columns:")
    print(subset_df.columns)

    print("\nSplit columns:")
    print(split_df.columns)

    print("\n📊 Official FunOTTA subset:")

    print(
        subset_df["diagnosis"].value_counts()
    )

    if "split" not in split_df.columns:

        raise ValueError(
            "split.csv must contain a 'split' column"
        )

    # =====================================================
    # MERGE SPLITS
    # =====================================================
    df = subset_df.merge(

        split_df[
            ["id_code", "split"]
        ],

        on="id_code",

        how="inner"
    )

    print(
        f"\n✅ Total merged samples: {len(df)}"
    )

    print("\nMerged columns:")
    print(df.columns)

    # =====================================================
    # SPLITS
    # =====================================================
    train_df = df[
        df["split"]
        .astype(str)
        .str.lower()
        .str.contains("train")
    ]

    val_df = df[
        df["split"]
        .astype(str)
        .str.lower()
        .str.contains("val")
    ]

    if len(train_df) == 0:

        raise ValueError(
            "No training samples found"
        )

    if len(val_df) == 0:

        raise ValueError(
            "No validation samples found"
        )

    print("\n📊 Train distribution:")

    print(
        train_df["diagnosis"].value_counts()
    )

    print("\n📊 Validation distribution:")

    print(
        val_df["diagnosis"].value_counts()
    )

    # =====================================================
    # DATASETS
    # =====================================================
    train_ds = EyePACSDataset(

        train_df,

        cfg.img_dir,

        get_transforms(
            cfg.img_size,
            "train"
        )
    )

    val_ds = EyePACSDataset(

        val_df,

        cfg.img_dir,

        get_transforms(
            cfg.img_size,
            "val"
        )
    )

    # =====================================================
    # SEEDED GENERATOR
    # =====================================================
    g = torch.Generator()

    g.manual_seed(
        cfg.seed
    )

    # =====================================================
    # TRAIN LOADER
    # =====================================================
    train_loader = DataLoader(

        train_ds,

        batch_size=cfg.batch_size,

        shuffle=True,

        num_workers=cfg.num_workers,

        pin_memory=cfg.pin_memory,

        persistent_workers=cfg.persistent_workers,

        prefetch_factor=cfg.prefetch_factor,

        drop_last=True,

        worker_init_fn=seed_worker,

        generator=g
    )

    # =====================================================
    # VAL LOADER
    # =====================================================
    val_loader = DataLoader(

        val_ds,

        batch_size=cfg.batch_size,

        shuffle=False,

        num_workers=cfg.num_workers,

        pin_memory=cfg.pin_memory,

        persistent_workers=cfg.persistent_workers,

        prefetch_factor=cfg.prefetch_factor,

        worker_init_fn=seed_worker,

        generator=g
    )

    print(
        f"\n✅ Train samples : {len(train_ds)}"
    )

    print(
        f"✅ Val samples   : {len(val_ds)}"
    )

    print(
        f"✅ Train batches : {len(train_loader)}"
    )

    print(
        f"✅ Val batches   : {len(val_loader)}"
    )

    return (
        train_loader,
        val_loader
    )



Writing /kaggle/working/dataset.py


In [10]:
%%writefile /kaggle/working/model.py

import torch
import torch.nn as nn
import torchvision.models as models


# =========================================================
# IMPROVED LONG-TRAINING RESNET50 DEEPSET MODEL
#
# OPTIMIZED FOR:
#
# ✅ Long 40-60 epoch training
# ✅ Better late-stage generalization
# ✅ Reduced overfitting
# ✅ Better TTA compatibility
# ✅ Stronger lesion localization
# ✅ Stable full-backbone fine-tuning
# ✅ Better cross-domain transfer
#
# NEW IMPROVEMENTS:
#
# ✅ Stronger feature regularization
# ✅ Better node-global fusion
# ✅ Residual fusion refinement
# ✅ Better attention stability
# ✅ Improved classifier robustness
# ✅ Cleaner gradient flow
# ✅ Better late-epoch stability
#
# IMPORTANT:
# This is an EVOLUTION of your good model,
# NOT a risky redesign.
# =========================================================


# =========================================================
# GeM POOLING
# =========================================================
class GeMPool2d(nn.Module):

    def __init__(
        self,
        p=3.0,
        eps=1e-6
    ):

        super().__init__()

        self.p = nn.Parameter(
            torch.ones(1) * p
        )

        self.eps = eps

    def forward(self, x):

        return torch.mean(

            x.clamp(min=self.eps).pow(self.p),

            dim=(-1, -2)

        ).pow(1.0 / self.p)


# =========================================================
# ATTENTION POOL
# =========================================================
class AttentionPool(nn.Module):

    def __init__(
        self,
        dim=128
    ):

        super().__init__()

        self.attn = nn.Sequential(

            nn.Linear(dim, 64),

            nn.GELU(),

            nn.Dropout(0.15),

            nn.Linear(64, 1)
        )

    def forward(
        self,
        nodes
    ):

        scores = self.attn(nodes)

        weights = torch.softmax(
            scores,
            dim=1
        )

        pooled = (
            weights * nodes
        ).sum(dim=1)

        return pooled


# =========================================================
# FUSION BLOCK
#
# improves feature interaction
# =========================================================
class FusionBlock(nn.Module):

    def __init__(
        self,
        dim=256
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Linear(dim, dim),

            nn.LayerNorm(dim),

            nn.GELU(),

            nn.Dropout(0.15),

            nn.Linear(dim, dim)
        )

        self.norm = nn.LayerNorm(dim)

    def forward(
        self,
        x
    ):

        residual = x

        x = self.block(x)

        x = x + residual

        x = self.norm(x)

        return x


# =========================================================
# MAIN MODEL
# =========================================================
class DeepSetModel(nn.Module):

    def __init__(
        self,
        num_classes=5
    ):

        super().__init__()

        self.num_classes = num_classes

        # =================================================
        # RESNET50
        # =================================================
        backbone = models.resnet50(

            weights=models.ResNet50_Weights.DEFAULT
        )

        self.features = nn.Sequential(
            *list(backbone.children())[:-2]
        )

        feat_dim = 2048

        # =================================================
        # GeM GLOBAL POOL
        # =================================================
        self.gem_pool = GeMPool2d()

        # =================================================
        # NODE PROJECTION
        # =================================================
        self.node_proj = nn.Sequential(

            nn.Linear(
                feat_dim,
                128
            ),

            nn.LayerNorm(128),

            nn.GELU(),

            nn.Dropout(0.15)
        )

        # =================================================
        # ATTENTION POOL
        # =================================================
        self.attention_pool = AttentionPool(
            dim=128
        )

        # =================================================
        # GLOBAL FEATURE PROJECTION
        # =================================================
        self.global_proj = nn.Sequential(

            nn.Linear(
                feat_dim,
                128
            ),

            nn.LayerNorm(128),

            nn.GELU(),

            nn.Dropout(0.15)
        )

        # =================================================
        # FEATURE FUSION
        # =================================================
        fused_dim = 256

        # =================================================
        # FUSION REFINEMENT
        # =================================================
        self.fusion_block = FusionBlock(
            dim=fused_dim
        )

        # =================================================
        # FEATURE DROPOUT
        # =================================================
        self.feature_dropout = nn.Dropout(
            0.25
        )

        # =================================================
        # CLASSIFIER
        #
        # improved long-training stability
        # =================================================
        self.classifier = nn.Sequential(

            nn.Linear(
                fused_dim,
                128
            ),

            nn.LayerNorm(128),

            nn.GELU(),

            nn.Dropout(0.25),

            nn.Linear(
                128,
                64
            ),

            nn.GELU(),

            nn.Dropout(0.15),

            nn.Linear(
                64,
                num_classes
            )
        )

        # =================================================
        # ORDINAL HEAD
        # =================================================
        self.ordinal_head = nn.Sequential(

            nn.Linear(
                fused_dim,
                64
            ),

            nn.GELU(),

            nn.Dropout(0.15),

            nn.Linear(
                64,
                num_classes - 1
            )
        )

        # =================================================
        # INIT
        # =================================================
        self._init_weights()

    # =====================================================
    # INIT
    # =====================================================
    def _init_weights(self):

        for m in self.modules():

            if isinstance(m, nn.Linear):

                nn.init.xavier_normal_(
                    m.weight
                )

                if m.bias is not None:

                    nn.init.zeros_(
                        m.bias
                    )

    # =====================================================
    # FORWARD
    # =====================================================
    def forward(
        self,
        x,
        return_nodes=True
    ):

        # =================================================
        # CNN FEATURES
        # =================================================
        feat = self.features(x)

        # =================================================
        # GLOBAL FEATURES
        # =================================================
        global_feat = self.gem_pool(
            feat
        )

        global_feat = self.global_proj(
            global_feat
        )

        # =================================================
        # GRID NODES
        # =================================================
        nodes = feat.flatten(2).transpose(
            1,
            2
        )

        # =================================================
        # NODE EMBEDDINGS
        # =================================================
        nodes = self.node_proj(
            nodes
        )

        # =================================================
        # ATTENTION NODE POOL
        # =================================================
        pooled_nodes = self.attention_pool(
            nodes
        )

        # =================================================
        # FUSION
        # =================================================
        fused = torch.cat(

            [
                pooled_nodes,
                global_feat
            ],

            dim=1
        )

        # =================================================
        # FUSION REFINEMENT
        # =================================================
        fused = self.fusion_block(
            fused
        )

        # =================================================
        # DROPOUT
        # =================================================
        fused = self.feature_dropout(
            fused
        )

        # =================================================
        # CLASSIFICATION
        # =================================================
        logits = self.classifier(
            fused
        )

        # =================================================
        # ORDINAL HEAD
        # =================================================
        ordinal_logits = self.ordinal_head(
            fused
        )

        # =================================================
        # RETURN
        # =================================================
        if return_nodes:

            return (

                logits,

                nodes,

                ordinal_logits
            )

        return logits


# =========================================================
# FACTORY
# =========================================================
def get_model(cfg):

    model = DeepSetModel(
        num_classes=cfg.num_classes
    )

    return model

Writing /kaggle/working/model.py


In [11]:
%%writefile /kaggle/working/train.py

import os
import torch
import torch.optim as optim
import torch.nn.functional as F
import random

from torch.amp import autocast, GradScaler

from tqdm import tqdm

import numpy as np

from sklearn.metrics import (

    cohen_kappa_score,

    accuracy_score,

    f1_score,

    roc_auc_score
)

from config import CFG
from dataset import get_loaders
from model import get_model


# =========================================================
# SEED
# =========================================================
def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


# =========================================================
# CUDA OPTIMIZATION
# =========================================================
torch.backends.cudnn.benchmark = False

torch.backends.cuda.matmul.allow_tf32 = True

torch.backends.cudnn.allow_tf32 = True

torch.set_float32_matmul_precision(
    "high"
)


# =========================================================
# EMA
# =========================================================
class EMA:

    def __init__(
        self,
        model,
        decay=0.999
    ):

        self.decay = decay

        self.shadow = {}

        self.backup = {}

        model = (
            model.module
            if isinstance(
                model,
                torch.nn.DataParallel
            )
            else model
        )

        for name, param in model.named_parameters():

            if param.requires_grad:

                self.shadow[name] = (
                    param.data.clone()
                )

    # =====================================================
    # FIXED EMA UPDATE
    #
    # handles newly unfrozen params
    # =====================================================
    def update(
        self,
        model
    ):

        model = (
            model.module
            if isinstance(
                model,
                torch.nn.DataParallel
            )
            else model
        )

        for name, param in model.named_parameters():

            if not param.requires_grad:
                continue

            # =============================================
            # NEW PARAM AFTER UNFREEZE
            # =============================================
            if name not in self.shadow:

                self.shadow[name] = (
                    param.data.clone()
                )

            # =============================================
            # EMA UPDATE
            # =============================================
            new_average = (

                self.decay *

                self.shadow[name]

                +

                (1.0 - self.decay) *

                param.data
            )

            self.shadow[name] = (
                new_average.clone()
            )

    def apply_shadow(
        self,
        model
    ):

        model = (
            model.module
            if isinstance(
                model,
                torch.nn.DataParallel
            )
            else model
        )

        for name, param in model.named_parameters():

            if not param.requires_grad:
                continue

            if name not in self.shadow:
                continue

            self.backup[name] = (
                param.data.clone()
            )

            param.data = (
                self.shadow[name]
            )

    def restore(
        self,
        model
    ):

        model = (
            model.module
            if isinstance(
                model,
                torch.nn.DataParallel
            )
            else model
        )

        for name, param in model.named_parameters():

            if name not in self.backup:
                continue

            param.data = (
                self.backup[name]
            )

        self.backup = {}


# =========================================================
# FOCAL LOSS
# =========================================================
class FocalLoss(torch.nn.Module):

    def __init__(
        self,
        gamma=1.8
    ):

        super().__init__()

        self.gamma = gamma

    def forward(
        self,
        logits,
        targets
    ):

        ce = F.cross_entropy(

            logits,

            targets,

            reduction="none"
        )

        pt = torch.exp(-ce)

        focal = (
            (1 - pt) ** self.gamma
        ) * ce

        return focal.mean()


# =========================================================
# ORDINAL TARGETS
# =========================================================
def ordinal_targets(

    labels,

    num_classes=5
):

    B = labels.size(0)

    targets = torch.zeros(

        B,

        num_classes - 1,

        device=labels.device
    )

    for i in range(num_classes - 1):

        targets[:, i] = (
            labels > i
        ).float()

    return targets


# =========================================================
# EXPECTED GRADE
# =========================================================
def expected_grade(logits):

    probs = F.softmax(
        logits,
        dim=1
    )

    grades = torch.arange(

        logits.size(1),

        device=logits.device

    ).float()

    pred = (
        probs * grades
    ).sum(dim=1)

    pred = torch.clamp(
        torch.round(pred),
        0,
        logits.size(1) - 1
    )

    return pred.long()


# =========================================================
# FREEZE BACKBONE
# =========================================================
def freeze_backbone(model):

    target_model = (

        model.module
        if isinstance(
            model,
            torch.nn.DataParallel
        )
        else model
    )

    for param in target_model.features.parameters():

        param.requires_grad = False

    print(
        "\n🧊 Backbone frozen"
    )


# =========================================================
# UNFREEZE LAYER4
# =========================================================
def unfreeze_layer4(model):

    target_model = (

        model.module
        if isinstance(
            model,
            torch.nn.DataParallel
        )
        else model
    )

    for name, param in target_model.features.named_parameters():

        if "7" in name:

            param.requires_grad = True

    print(
        "\n🔥 Layer4 unfrozen"
    )


# =========================================================
# FULL BACKBONE
# =========================================================
def unfreeze_full_backbone(model):

    target_model = (

        model.module
        if isinstance(
            model,
            torch.nn.DataParallel
        )
        else model
    )

    for param in target_model.features.parameters():

        param.requires_grad = True

    print(
        "\n🚀 Full backbone unfrozen"
    )


# =========================================================
# VALIDATION
# =========================================================
@torch.no_grad()
def validate(

    model,

    loader,

    device,

    cfg,

    ce_criterion,

    focal_criterion
):

    model.eval()

    preds = []

    labels_all = []

    probs_all = []

    total_loss = 0.0

    for images, labels in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        with autocast(

            device_type="cuda",

            enabled=cfg.use_amp
        ):

            logits, _, ordinal_logits = model(
                images
            )

            ce_loss = ce_criterion(
                logits,
                labels
            )

            focal_loss = focal_criterion(
                logits,
                labels
            )

            ord_targets = ordinal_targets(
                labels,
                cfg.num_classes
            )

            ord_loss = (
                F.binary_cross_entropy_with_logits(

                    ordinal_logits,

                    ord_targets
                )
            )

            loss = (

                0.70 * ce_loss

                +

                0.30 * focal_loss

                +

                cfg.ordinal_weight * ord_loss
            )

        total_loss += loss.item()

        pred = expected_grade(
            logits
        )

        probs = F.softmax(
            logits,
            dim=1
        )

        preds.extend(
            pred.cpu().numpy()
        )

        labels_all.extend(
            labels.cpu().numpy()
        )

        probs_all.extend(
            probs.cpu().numpy()
        )

    preds = np.array(preds)

    labels_all = np.array(labels_all)

    probs_all = np.array(probs_all)

    acc = accuracy_score(
        labels_all,
        preds
    )

    f1 = f1_score(
        labels_all,
        preds,
        average="macro"
    )

    qwk = cohen_kappa_score(
        labels_all,
        preds,
        weights="quadratic"
    )

    try:

        auc = roc_auc_score(

            labels_all,

            probs_all,

            multi_class="ovr"
        )

    except:

        auc = 0.0

    val_loss = (
        total_loss / len(loader)
    )

    return (

        val_loss,

        acc,

        f1,

        qwk,

        auc
    )


# =========================================================
# MAIN
# =========================================================
def main():

    cfg = CFG()

    set_seed(cfg.seed)

    print(
        f"\n🎲 Seed: {cfg.seed}"
    )

    device = cfg.device

    print(
        f"💾 Save Path: {cfg.save_path}"
    )

    gpu_count = torch.cuda.device_count()

    if torch.cuda.is_available():

        print(
            f"🔥 GPUs: {gpu_count}"
        )

    # =====================================================
    # DATA
    # =====================================================
    train_loader, val_loader = get_loaders(cfg)

    # =====================================================
    # MODEL
    # =====================================================
    model = get_model(cfg)

    model = model.to(device)

    # =====================================================
    # MULTI GPU
    # =====================================================
    if gpu_count > 1:

        print(
            f"\n🚀 Using {gpu_count} GPUs"
        )

        model = torch.nn.DataParallel(
            model
        )

    # =====================================================
    # FREEZE STAGE
    # =====================================================
    freeze_backbone(model)

    # =====================================================
    # EMA
    # =====================================================
    ema = EMA(
        model,
        decay=0.999
    )

    # =====================================================
    # LOSS
    # =====================================================
    ce_criterion = torch.nn.CrossEntropyLoss(

        label_smoothing=cfg.label_smoothing
    )

    focal_criterion = FocalLoss(
        gamma=1.8
    )

    # =====================================================
    # OPTIMIZER
    # =====================================================
    optimizer = optim.AdamW(

        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),

        lr=cfg.lr,

        weight_decay=cfg.weight_decay,

        fused=True
    )

    # =====================================================
    # SCHEDULER
    # =====================================================
    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(

            optimizer,

            T_0=10,

            T_mult=2,

            eta_min=1e-6
        )
    )

    # =====================================================
    # AMP
    # =====================================================
    scaler = GradScaler(

        "cuda",

        enabled=cfg.use_amp
    )

    # =====================================================
    # BEST
    # =====================================================
    best_score = -1.0

    best_qwk = -1.0

    best_f1 = -1.0

    best_auc = -1.0

    # =====================================================
    # PATIENCE
    # =====================================================
    patience = 15

    patience_counter = 0

    print(
        "\n🔥 Improved Long Training Started\n"
    )

    # =====================================================
    # EPOCHS
    # =====================================================
    for epoch in range(cfg.epochs):

        # =================================================
        # STAGE 2
        # =================================================
        if epoch == 5:

            unfreeze_layer4(model)

            optimizer = optim.AdamW(

                filter(
                    lambda p: p.requires_grad,
                    model.parameters()
                ),

                lr=8e-5,

                weight_decay=cfg.weight_decay,

                fused=True
            )

            print(
                "\n🚀 Stage 2 Fine-tuning"
            )

        # =================================================
        # STAGE 3
        # =================================================
        if epoch == 15:

            unfreeze_full_backbone(model)

            optimizer = optim.AdamW(

                model.parameters(),

                lr=2e-5,

                weight_decay=cfg.weight_decay,

                fused=True
            )

            print(
                "\n🔥 Full Fine-tuning Stage"
            )

        model.train()

        running_loss = 0.0

        loop = tqdm(

            train_loader,

            leave=False
        )

        for images, labels in loop:

            images = images.to(

                device,

                non_blocking=True
            )

            labels = labels.to(

                device,

                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with autocast(

                device_type="cuda",

                enabled=cfg.use_amp
            ):

                logits, _, ordinal_logits = model(
                    images
                )

                ce_loss = ce_criterion(
                    logits,
                    labels
                )

                focal_loss = focal_criterion(
                    logits,
                    labels
                )

                ord_targets = ordinal_targets(
                    labels,
                    cfg.num_classes
                )

                ord_loss = (
                    F.binary_cross_entropy_with_logits(

                        ordinal_logits,

                        ord_targets
                    )
                )

                loss = (

                    0.70 * ce_loss

                    +

                    0.30 * focal_loss

                    +

                    cfg.ordinal_weight * ord_loss
                )

            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                cfg.grad_clip
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            # =============================================
            # EMA UPDATE
            # =============================================
            ema.update(model)

            running_loss += loss.item()

            loop.set_postfix(

                loss=f"{loss.item():.4f}"
            )

        scheduler.step(epoch + 1)

        # =================================================
        # EMA VALIDATION
        # =================================================
        ema.apply_shadow(model)

        (
            val_loss,
            acc,
            f1,
            qwk,
            auc

        ) = validate(

            model,

            val_loader,

            device,

            cfg,

            ce_criterion,

            focal_criterion
        )

        ema.restore(model)

        train_loss = (
            running_loss / len(train_loader)
        )

        # =================================================
        # FINAL SCORE
        # =================================================
        score = (

            0.50 * f1

            +

            0.50 * qwk
        )

        current_lr = optimizer.param_groups[0]["lr"]

        print(

            f"Epoch {epoch+1:02d} | "

            f"LR {current_lr:.6f} | "

            f"Train {train_loss:.4f} | "

            f"Val {val_loss:.4f} | "

            f"Acc {acc:.4f} | "

            f"F1 {f1:.4f} | "

            f"QWK {qwk:.4f} | "

            f"AUC {auc:.4f}"
        )

        # =================================================
        # SAVE BEST
        # =================================================
        if score > best_score:

            best_score = score

            best_qwk = qwk

            best_f1 = f1

            best_auc = auc

            patience_counter = 0

            save_model = (

                model.module

                if isinstance(
                    model,
                    torch.nn.DataParallel
                )

                else model
            )

            torch.save(
            {
                "seed": cfg.seed,
                "model": save_model.state_dict(),
                "best_score": best_score,
                "best_f1": best_f1,
                "best_qwk": best_qwk,
                "best_auc": best_auc,
            },
            cfg.save_path
            )

            print(

                f"✅ Saved Best | "

                f"Score {score:.4f} | "

                f"F1 {f1:.4f} | "

                f"QWK {qwk:.4f}"
            )

        else:

            patience_counter += 1

            print(

                f"⏳ Patience "

                f"{patience_counter}/{patience}"
            )

        # =================================================
        # EARLY STOPPING
        # =================================================
        if patience_counter >= patience:

            print(
                "\n🛑 Early stopping"
            )

            break

    # =====================================================
    # FINAL
    # =====================================================
    print(

    "\n🏁 Training Completed"

    )
    
    print(
    
        f"🎲 Seed Used: {cfg.seed}"
    
    )
    print(
        f"🏆 Best F1  : {best_f1:.4f}"
    )

    print(
        f"🏆 Best QWK : {best_qwk:.4f}"
    )

    print(
        f"🏆 Best AUC : {best_auc:.4f}"
    )


# =========================================================
# ENTRY
# =========================================================
if __name__ == "__main__":

    main()

Writing /kaggle/working/train.py


In [9]:
#!python /kaggle/working/train.py


🎲 Seed: 2026
💾 Save Path: /kaggle/working/best_dsr50_balanced_seed_2026.pth
🔥 GPUs: 2

Subset columns:
Index(['id_code', 'diagnosis'], dtype='object')

Split columns:
Index(['id_code', 'diagnosis', 'split'], dtype='object')

📊 Official FunOTTA subset:
diagnosis
1    708
0    708
2    708
4    708
3    708
Name: count, dtype: int64

✅ Total merged samples: 3540

Merged columns:
Index(['id_code', 'diagnosis', 'split'], dtype='object')

📊 Train distribution:
diagnosis
2    496
1    496
4    496
0    495
3    495
Name: count, dtype: int64

📊 Validation distribution:
diagnosis
3    107
1    106
4    106
2    106
0    106
Name: count, dtype: int64

✅ Train samples : 2478
✅ Val samples   : 531
✅ Train batches : 154
✅ Val batches   : 34
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 213MB/s]

🚀 Using 2 GPUs

🧊 Backbone frozen

🔥 Improved

In [12]:
import torch
from model import get_model
from config import CFG

cfg = CFG()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = get_model(cfg).to(device)

checkpoint = torch.load(
    "/kaggle/input/models/knightcracker/seed2026-model/pytorch/default/1/best_dsr50_balanced_seed_2026.pth",
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model"]
)

model.eval()

print("✅ Trained model loaded")
print("Seed:", checkpoint["seed"])
print("Best F1:", checkpoint["best_f1"])
print("Best QWK:", checkpoint["best_qwk"])
print("Best AUC:", checkpoint["best_auc"])

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 217MB/s]


✅ Trained model loaded
Seed: 2026
Best F1: 0.4948352871064026
Best QWK: 0.7444184875641929
Best AUC: 0.0


In [13]:
%%writefile /kaggle/working/baseline.py

import torch
import torch.nn.functional as F
import numpy as np


# =========================================================
# EXPECTED GRADE
# =========================================================
def expected_grade(logits):

    probs = F.softmax(
        logits,
        dim=-1
    )

    grades = torch.arange(
        logits.size(-1),
        device=logits.device
    ).float()

    pred = (
        probs * grades
    ).sum(dim=-1)

    return pred


# =========================================================
# BASELINE
# =========================================================
@torch.no_grad()
def run_baseline(

    model,

    loader,

    return_probs=False
):

    model.eval()

    device = next(
        model.parameters()
    ).device

    preds = []

    probs_all = []

    labels_all = []

    for images, labels in loader:

        images = images.to(device)

        out = model(images)

        # =============================================
        # SAFE MODEL OUTPUT
        # =============================================
        if isinstance(out, tuple):

            logits = out[0]

        else:

            logits = out

        # =============================================
        # PROBS
        # =============================================
        probs = F.softmax(
            logits,
            dim=-1
        )

        # =============================================
        # EXPECTED GRADE
        # =============================================
        pred = expected_grade(
            logits
        )

        preds.extend(
            pred.cpu().numpy()
        )

        labels_all.extend(
            labels.numpy()
        )

        if return_probs:

            probs_all.extend(
                probs.cpu().numpy()
            )

    preds = np.array(
        preds,
        dtype=np.float32
    )

    labels_all = np.array(
        labels_all,
        dtype=np.int64
    )

    # =====================================================
    # RETURN WITH PROBS
    # =====================================================
    if return_probs:

        probs_all = np.array(
            probs_all,
            dtype=np.float32
        )

        return (
            preds,
            labels_all,
            probs_all
        )

    # =====================================================
    # RETURN NORMAL
    # =====================================================
    return (
        preds,
        labels_all
    )

Writing /kaggle/working/baseline.py


In [14]:
%%writefile /kaggle/working/metrics.py

import numpy as np

from sklearn.metrics import (

    accuracy_score,

    f1_score,

    cohen_kappa_score,

    roc_auc_score
)

from sklearn.preprocessing import label_binarize


# =========================================================
# FINAL REVIEWER-SAFE METRICS
#
# IEEE / MICCAI / Medical Imaging Safe
#
# Features:
#
# ✅ Correct multiclass AUC
# ✅ Safe probability normalization
# ✅ Handles missing classes safely
# ✅ Stable under extreme imbalance
# ✅ Supports DR ordinal grading
# ✅ Works for 5-seed evaluation
# ✅ CSV-safe numeric outputs
#
# =========================================================


# =========================================================
# SAFE NORMALIZE
# =========================================================
def safe_normalize_probs(probs):

    probs = np.asarray(
        probs,
        dtype=np.float32
    )

    probs = np.nan_to_num(

        probs,

        nan=1e-8,

        posinf=1e-8,

        neginf=1e-8
    )

    probs = np.clip(

        probs,

        1e-8,

        1.0
    )

    row_sum = probs.sum(

        axis=1,

        keepdims=True
    )

    probs = probs / (

        row_sum + 1e-8
    )

    return probs


# =========================================================
# SAFE MULTICLASS AUC
# =========================================================
def compute_auc(

    probs,

    labels,

    num_classes=5
):
    """
    Reviewer-safe multiclass AUC.

    Parameters
    ----------
    probs : ndarray [N, C]

    labels : ndarray [N]

    Returns
    -------
    float
    """

    probs = safe_normalize_probs(probs)

    labels = np.asarray(

        labels,

        dtype=np.int64
    )

    # =====================================================
    # SHAPE CHECK
    # =====================================================
    if probs.ndim != 2:

        raise ValueError(

            f"Expected probs [N,C], got {probs.shape}"
        )

    if probs.shape[1] != num_classes:

        raise ValueError(

            f"Expected {num_classes} classes, "

            f"got {probs.shape[1]}"
        )

    # =====================================================
    # ONE HOT LABELS
    # =====================================================
    labels_onehot = label_binarize(

        labels,

        classes=list(range(num_classes))
    )

    # =====================================================
    # HANDLE MISSING CLASSES
    #
    # Important for small datasets
    # =====================================================
    valid_classes = []

    for c in range(num_classes):

        unique_vals = np.unique(

            labels_onehot[:, c]
        )

        if len(unique_vals) > 1:

            valid_classes.append(c)

    if len(valid_classes) == 0:

        return None

    labels_valid = labels_onehot[:, valid_classes]

    probs_valid = probs[:, valid_classes]

    # =====================================================
    # COMPUTE OVR MACRO AUC
    # =====================================================
    auc = roc_auc_score(

        labels_valid,

        probs_valid,

        multi_class="ovr",

        average="macro"
    )

    return float(auc)


# =========================================================
# OPTIONAL THRESHOLD SEARCH
# =========================================================
def optimize_thresholds(

    preds,

    labels,

    mode="balanced"
):
    """
    Threshold optimization
    for ordinal DR grading.

    Use ONLY on validation/calibration set.
    """

    preds = np.asarray(

        preds,

        dtype=np.float32
    )

    labels = np.asarray(

        labels,

        dtype=np.int64
    )

    best_score = -1.0

    best_th = [

        0.5,

        1.5,

        2.5,

        3.5
    ]

    # =====================================================
    # GRID SEARCH
    # =====================================================
    for t1 in np.linspace(0.6, 0.9, 5):

        for t2 in np.linspace(1.4, 2.2, 6):

            if t2 <= t1:
                continue

            for t3 in np.linspace(2.2, 3.0, 6):

                if t3 <= t2:
                    continue

                for t4 in np.linspace(3.0, 3.8, 5):

                    if t4 <= t3:
                        continue

                    thresholds = [

                        t1,

                        t2,

                        t3,

                        t4
                    ]

                    preds_cls = np.digitize(

                        preds,

                        bins=thresholds
                    )

                    f1 = f1_score(

                        labels,

                        preds_cls,

                        average="macro",

                        zero_division=0
                    )

                    qwk = cohen_kappa_score(

                        labels,

                        preds_cls,

                        weights="quadratic"
                    )

                    # =====================================
                    # OBJECTIVE
                    # =====================================
                    if mode == "f1":

                        score = f1

                    else:

                        score = (

                            0.5 * f1

                            +

                            0.5 * qwk
                        )

                    # =====================================
                    # UPDATE
                    # =====================================
                    if score > best_score:

                        best_score = score

                        best_th = thresholds

    print(

        f"🔥 Best thresholds: {best_th}"
    )

    print(

        f"🔥 Best score: {best_score:.4f}"
    )

    return best_th


# =========================================================
# MAIN METRICS
# =========================================================
def compute_metrics_from_preds(

    preds,

    labels,

    probs=None,

    ranking_scores=None,

    thresholds=None
):
    """
    Main evaluation function.

    Parameters
    ----------
    preds : ndarray

    labels : ndarray

    probs : ndarray [N,C]

    ranking_scores : optional
        internal signal only

    thresholds : optional
        ordinal threshold bins
    """

    # =====================================================
    # SAFE CONVERSION
    # =====================================================
    preds = np.asarray(

        preds,

        dtype=np.int64
    )

    labels = np.asarray(

        labels,

        dtype=np.int64
    )

    # =====================================================
    # OPTIONAL THRESHOLDING
    # =====================================================
    if thresholds is not None:

        preds_cls = np.digitize(

            preds,

            bins=thresholds
        )

    else:

        preds_cls = preds

    # =====================================================
    # ACCURACY
    # =====================================================
    acc = accuracy_score(

        labels,

        preds_cls
    )

    # =====================================================
    # MACRO F1
    # =====================================================
    f1 = f1_score(

        labels,

        preds_cls,

        average="macro",

        zero_division=0
    )

    # =====================================================
    # QUADRATIC WEIGHTED KAPPA
    # =====================================================
    qwk = cohen_kappa_score(

        labels,

        preds_cls,

        weights="quadratic"
    )

    # =====================================================
    # AUC
    #
    # IMPORTANT:
    # Uses ONLY probabilities.
    #
    # ranking_scores are NEVER used
    # directly for official AUC.
    # =====================================================
    auc = None

    if probs is not None:

        try:

            auc = compute_auc(

                probs,

                labels,

                num_classes=5
            )

        except Exception as e:

            print(

                f"⚠ AUC computation failed: {e}"
            )

            auc = None

    # =====================================================
    # FINAL OUTPUT
    # =====================================================
    results = {

        "accuracy": float(acc),

        "f1": float(f1),

        "qwk": float(qwk),

        "auc": (
            None
            if auc is None
            else float(auc)
        )
    }

    return results

Writing /kaggle/working/metrics.py


In [15]:
%%writefile /kaggle/working/dataset_unified.py

import os
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader

from PIL import Image

import torchvision.transforms as T

from sklearn.model_selection import train_test_split


# =========================================================
# GENERIC DATASET
# =========================================================
class DRDataset(Dataset):

    def __init__(
        self,
        df,
        img_dir,
        img_size=224
    ):

        self.df = df.reset_index(drop=True)

        self.img_dir = img_dir

        self.transform = T.Compose([

            T.Resize((img_size, img_size)),

            T.ToTensor(),

            T.Normalize(
                [0.485,0.456,0.406],
                [0.229,0.224,0.225]
            )
        ])

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        base = os.path.join(
            self.img_dir,
            row["image"]
        )

        # =================================================
        # IMAGE PATH FIXES
        # =================================================
        possible_paths = [

            base,

            base.replace(".jpg", ".JPG"),

            base.replace(".png", ".JPG"),

            base.replace(".jpeg", ".jpg"),

            base.replace(".jpeg", ".png"),

            base.replace(".png", ".jpg"),
        ]

        img_path = None

        for p in possible_paths:

            if os.path.exists(p):

                img_path = p
                break

        if img_path is None:

            raise FileNotFoundError(
                f"Missing image: {base}"
            )

        image = Image.open(
            img_path
        ).convert("RGB")

        image = self.transform(image)

        label = int(row["label"])

        return image, label


# =========================================================
# EYEPACS VAL
# =========================================================
def get_eyepacs_val_loader(cfg):

    df = pd.read_csv(
        cfg.labels_path
    )

    df = df.rename(columns={

        "image": "id_code",

        "level": "diagnosis"
    })

    df["image"] = (
        df["id_code"] + ".jpeg"
    )

    df["label"] = (
        df["diagnosis"]
    )

    _, val_df = train_test_split(

        df,

        test_size=0.2,

        stratify=df["label"],

        random_state=cfg.seed
    )

    return DataLoader(

        DRDataset(
            val_df,
            cfg.img_dir,
            cfg.img_size
        ),

        batch_size=cfg.batch_size,

        shuffle=False,

        num_workers=cfg.num_workers
    )


# =========================================================
# APTOS
# =========================================================
def get_aptos_loader(cfg):

    df = pd.read_csv(
        "/kaggle/input/datasets/mariaherrerot/aptos2019/train_1.csv"
    )

    df["image"] = (
        df["id_code"] + ".png"
    )

    df["label"] = (
        df["diagnosis"]
    )

    return DataLoader(

        DRDataset(
            df,
            "/kaggle/input/datasets/mariaherrerot/aptos2019/train_images/train_images",
            cfg.img_size
        ),

        batch_size=cfg.batch_size,

        shuffle=False,

        num_workers=cfg.num_workers
    )


# =========================================================
# IDRID
# =========================================================
def get_idrid_loader(cfg):

    df = pd.read_csv(
        "/kaggle/input/datasets/mariaherrerot/idrid-dataset/idrid_labels.csv"
    )

    df = df[
        ["id_code", "diagnosis"]
    ].dropna()

    df["image"] = (
        df["id_code"] + ".jpg"
    )

    df["label"] = (
        df["diagnosis"]
    )

    return DataLoader(

        DRDataset(
            df,
            "/kaggle/input/datasets/mariaherrerot/idrid-dataset/Imagenes/Imagenes",
            cfg.img_size
        ),

        batch_size=cfg.batch_size,

        shuffle=False,

        num_workers=cfg.num_workers
    )


# =========================================================
# MESSIDOR
# =========================================================
def get_messidor_loader(cfg):

    df = pd.read_csv(

        "/kaggle/input/datasets/"
        "mariaherrerot/"
        "messidor2preprocess/"
        "messidor_data.csv"
    )

    df = df[
        df["adjudicated_gradable"] == 1
    ]

    df["image"] = (
        df["id_code"]
    )

    df["label"] = (
        df["diagnosis"]
        .astype(int)
    )

    return DataLoader(

        DRDataset(
            df,
            "/kaggle/input/datasets/"
            "mariaherrerot/"
            "messidor2preprocess/"
            "messidor-2/messidor-2/preprocess",
            cfg.img_size
        ),

        batch_size=cfg.batch_size,

        shuffle=False,

        num_workers=cfg.num_workers
    )


# =========================================================
# FGADR
# =========================================================
def get_fgadr_loader(cfg):

    base_dir = (

        "/kaggle/input/datasets/"
        "kanojiashashwat/"
        "fgadr-dataset/"
        "Seg-set"
    )

    csv_path = os.path.join(

        base_dir,

        "DR_Seg_Grading_Label.csv"
    )

    img_dir = os.path.join(

        base_dir,

        "Original_Images"
    )

    # =====================================================
    # LOAD CSV (NO HEADER)
    # =====================================================
    df = pd.read_csv(

        csv_path,

        header=None
    )

    # =====================================================
    # STANDARDIZE
    # =====================================================
    df.columns = [

        "image",

        "label"
    ]

    df["image"] = (

        df["image"]

        .astype(str)

        .str.strip()
    )

    df["label"] = (

        df["label"]

        .astype(int)
    )

    # =====================================================
    # FILTER LABELS
    # =====================================================
    df = df[
        df["label"].isin([0,1,2,3,4])
    ]

    # =====================================================
    # DEBUG
    # =====================================================
    print(
        f"\n✅ FGADR samples: {len(df)}"
    )

    print(
        "\nFGADR Label Distribution:"
    )

    print(

        df["label"]

        .value_counts()

        .sort_index()
    )

    print(
        "\nFGADR Example Images:"
    )

    print(
        df["image"].head()
    )

    return DataLoader(

        DRDataset(
            df,
            img_dir,
            cfg.img_size
        ),

        batch_size=cfg.batch_size,

        shuffle=False,

        num_workers=cfg.num_workers
    )


# =========================================================
# DDR
# =========================================================
def get_ddr_loader(cfg):

    csv_path = (
        "/kaggle/input/datasets/"
        "mariaherrerot/"
        "ddrdataset/"
        "DR_grading.csv"
    )

    img_root = (
        "/kaggle/input/datasets/"
        "mariaherrerot/"
        "ddrdataset/"
        "DR_grading"
    )

    df = pd.read_csv(csv_path)

    df["image"] = df["id_code"]

    df["label"] = df["diagnosis"]

    # =====================================================
    # BUILD PATH MAP
    # =====================================================
    all_paths = {}

    for root, _, files in os.walk(img_root):

        for f in files:

            all_paths[f] = os.path.join(
                root,
                f
            )

    # =====================================================
    # DATASET
    # =====================================================
    class DDRDatasetFixed(Dataset):

        def __init__(
            self,
            df,
            path_map,
            img_size=224
        ):

            self.df = df.reset_index(drop=True)

            self.path_map = path_map

            self.transform = T.Compose([

                T.Resize((img_size, img_size)),

                T.ToTensor(),

                T.Normalize(
                    [0.485,0.456,0.406],
                    [0.229,0.224,0.225]
                )
            ])

        def __len__(self):

            return len(self.df)

        def __getitem__(self, idx):

            row = self.df.iloc[idx]

            fname = row["image"]

            if fname not in self.path_map:

                raise FileNotFoundError(fname)

            img_path = self.path_map[fname]

            image = Image.open(
                img_path
            ).convert("RGB")

            image = self.transform(image)

            label = int(row["label"])

            return image, label

    dataset = DDRDatasetFixed(

        df,

        all_paths,

        cfg.img_size
    )

    return DataLoader(

        dataset,

        batch_size=cfg.batch_size,

        shuffle=False,

        num_workers=cfg.num_workers
    )


# =========================================================
# DEEPDRID
# =========================================================
def get_deepdrid_loader(cfg):

    base_dir = (

        "/kaggle/input/datasets/"
        "yoctoman/"
        "deepdrid/"
        "DeepDRiD-master/"
        "regular_fundus_images"
    )

    csv_path = os.path.join(

        base_dir,

        "regular-fundus-validation/"
        "regular-fundus-validation.csv"
    )

    df = pd.read_csv(csv_path)

    df["label"] = (
        df["patient_DR_Level"]
    )

    df["image"] = df["image_path"].apply(

        lambda x: os.path.basename(
            x.replace("\\", "/")
        )
    )

    # =====================================================
    # PATH MAP
    # =====================================================
    path_map = {}

    for root, _, files in os.walk(base_dir):

        for f in files:

            path_map[f] = os.path.join(
                root,
                f
            )

    # =====================================================
    # DATASET
    # =====================================================
    class DeepDRiDFixed(Dataset):

        def __init__(
            self,
            df,
            path_map,
            img_size=224
        ):

            self.df = df.reset_index(drop=True)

            self.path_map = path_map

            self.transform = T.Compose([

                T.Resize((img_size, img_size)),

                T.ToTensor(),

                T.Normalize(
                    [0.485,0.456,0.406],
                    [0.229,0.224,0.225]
                )
            ])

        def __len__(self):

            return len(self.df)

        def __getitem__(self, idx):

            row = self.df.iloc[idx]

            fname = row["image"]

            if fname not in self.path_map:

                raise FileNotFoundError(
                    f"Missing: {fname}"
                )

            img_path = self.path_map[fname]

            image = Image.open(
                img_path
            ).convert("RGB")

            image = self.transform(image)

            label = int(row["label"])

            return image, label

    dataset = DeepDRiDFixed(

        df,

        path_map,

        cfg.img_size
    )

    return DataLoader(

        dataset,

        batch_size=cfg.batch_size,

        shuffle=False,

        num_workers=cfg.num_workers
    )

Writing /kaggle/working/dataset_unified.py


In [16]:
# =========================================================
# COMPUTE EYEPACS EMPIRICAL SOURCE PRIOR
#
# Reads the full EyePACS training labels and computes the
# class-frequency prior used by CF-NODE source-prior
# correction.  Output is printed in two forms:
#
#   (a) a static list to paste into config.py
#   (b) a runtime one-liner for run.py
# =========================================================

import pandas as pd
import numpy as np
from config import CFG

# ---------------------------------------------------------
# LOAD LABELS
# ---------------------------------------------------------
df = pd.read_csv(CFG.labels_path)

# ---------------------------------------------------------
# DETECT LABEL COLUMN
#
# EyePACS standard schema is (image, level).
# Falls back to the second column for safety.
# ---------------------------------------------------------
if "level" in df.columns:

    label_col = "level"

elif "diagnosis" in df.columns:

    label_col = "diagnosis"

else:

    label_col = df.columns[1]

# ---------------------------------------------------------
# RESTRICT TO VALID GRADES  (0..num_classes-1)
# ---------------------------------------------------------
df = df[
    df[label_col].between(0, CFG.num_classes - 1)
].copy()

# ---------------------------------------------------------
# COUNT
# ---------------------------------------------------------
counts = (
    df[label_col]
    .value_counts()
    .reindex(range(CFG.num_classes), fill_value=0)
    .sort_index()
)

total = int(counts.sum())

prior = (counts.astype(float) / max(total, 1)).tolist()

# ---------------------------------------------------------
# REPORT
# ---------------------------------------------------------
print("=" * 56)
print(" EyePACS Empirical Class Prior")
print(" Source: " + CFG.labels_path)
print(" Label column: " + label_col)
print("=" * 56)

print(f" {'Grade':<8}{'Count':>10}{'Frequency':>14}{'Percent':>12}")
print(" " + "-" * 54)

for k, c in counts.items():

    f = c / max(total, 1)

    print(
        f" {k:<8}{c:>10,d}{f:>14.6f}{f*100:>11.4f}%"
    )

print(" " + "-" * 54)
print(f" {'Total':<8}{total:>10,d}")
print()

# ---------------------------------------------------------
# PASTE-INTO-CONFIG FORM
# ---------------------------------------------------------
print("=" * 56)
print(" (a) Paste this into config.py (CFG.source_prior):")
print("=" * 56)
print()
print("    source_prior = [")

for p in prior:

    print(f"        {p:.6f},")

print("    ]")
print()

# ---------------------------------------------------------
# RUNTIME ONE-LINER
# ---------------------------------------------------------
print("=" * 56)
print(" (b) Or set it dynamically in run.py / your notebook:")
print("=" * 56)
print()
print(f"    CFG.source_prior = {[round(p, 6) for p in prior]}")
print()

# ---------------------------------------------------------
# VERIFY  ( sums to 1 )
# ---------------------------------------------------------
assert abs(sum(prior) - 1.0) < 1e-6, "prior does not sum to 1"

print("✓ Prior verified — sums to 1.")



 EyePACS Empirical Class Prior
 Source: /kaggle/input/datasets/dreamer07/eyepacs/trainLabels.csv/trainLabels.csv
 Label column: level
 Grade        Count     Frequency     Percent
 ------------------------------------------------------
 0           25,810      0.734783    73.4783%
 1            2,443      0.069550     6.9550%
 2            5,292      0.150658    15.0658%
 3              873      0.024853     2.4853%
 4              708      0.020156     2.0156%
 ------------------------------------------------------
 Total       35,126

 (a) Paste this into config.py (CFG.source_prior):

    source_prior = [
        0.734783,
        0.069550,
        0.150658,
        0.024853,
        0.020156,
    ]

 (b) Or set it dynamically in run.py / your notebook:

    CFG.source_prior = [0.734783, 0.06955, 0.150658, 0.024853, 0.020156]

✓ Prior verified — sums to 1.


In [17]:
%%writefile /kaggle/working/tta_node.py
# =========================================================
# CAUSAL ORDINAL NODE TTA  —  CF-NODE
#
# IEEE Access submission, 2026.
#
# RESTORED v1 ALGORITHM (the configuration that produced
# the dominant Acc / F1 / AUC results on the 708-per-class
# balanced source model):
#
#   * EyePACS source prior  (0.73, 0.07, 0.15, 0.025, 0.025)
#   * Prior correction strength  0.18
#   * Causal-gated ordinal blend
#       alpha = 0.05 g + 0.55 g (1 - conf)
#       capped at 0.65
#   * argmax point estimate
#
# This file adds three NON-BEHAVIOURAL safety patches:
#
#   1. cf_strength may be a scalar OR a per-target dict.
#      When the resolved value is 1.0 the behaviour is
#      identical to v1.  (v1 ignored the dict entirely.)
#
#   2. cf_levels = None in cfg is safely treated as
#      "use the default 4-scale schedule".
#
#   3. The cf_strength resolver also reads cfg.target_dataset
#      so per-target tuning works without touching run.py.
#
# REFERENCES
#   [1] Wang et al., "TENT", ICLR 2021.
#   [2] Niu et al., "SAR", ICLR 2023.
#   [3] Niu et al., "EATA", ICML 2022.
#   [4] Iwasawa & Matsuo, "T3A", NeurIPS 2021.
# =========================================================

import torch
import torch.nn.functional as F
import numpy as np

# =========================================================
# DEFAULT SOURCE PRIOR
#
# EyePACS empirical prior — the working v1 configuration.
# Override via cfg.source_prior if needed.
# =========================================================
DEFAULT_SOURCE_PRIOR = torch.tensor([
    0.73,
    0.07,
    0.15,
    0.025,
    0.025,
]).float()

# =========================================================
# COUNTERFACTUAL SUPPRESSION SCHEDULE
# =========================================================
DEFAULT_CF_LEVELS = (

    (0.02, 0.75),

    (0.05, 0.40),

    (0.10, 0.20),

    (0.15, 0.10),
)

# =========================================================
# AUGMENTATIONS
# =========================================================
AUGMENTATIONS = [

    lambda x: x,

    lambda x: torch.flip(x, dims=[3]),

    lambda x: torch.flip(x, dims=[2]),

    lambda x: torch.rot90(x, k=1, dims=[2, 3]),

    lambda x: torch.rot90(x, k=3, dims=[2, 3]),
]

# =========================================================
# CFG ACCESSOR
# =========================================================
def _cfg(cfg, key, default):

    return getattr(cfg, key, default)

# =========================================================
# RESOLVE CF STRENGTH
#
# Default 1.0 → v1 behaviour preserved.
# =========================================================
def _resolve_cf_strength(cfg):

    cf = _cfg(cfg, 'cf_strength', 1.0)

    if isinstance(cf, dict):

        name = (
            _cfg(cfg, 'target_name',    None)
            or _cfg(cfg, 'target_dataset', None)
            or _cfg(cfg, 'dataset',     None)
            or _cfg(cfg, 'target',      None)
            or _cfg(cfg, 'domain',      None)
        )

        if isinstance(name, str):

            name = name.lower()

        if name is not None and name in cf:

            cf = cf[name]

        elif 'default' in cf:

            cf = cf['default']

        else:

            cf = 1.0

    return float(cf)

# =========================================================
# RESOLVE SOURCE PRIOR
# =========================================================
def _resolve_source_prior(cfg, device):

    prior = _cfg(cfg, 'source_prior', None)

    if prior is None:

        prior = DEFAULT_SOURCE_PRIOR

    if not torch.is_tensor(prior):

        prior = torch.tensor(
            prior,
            dtype=torch.float32
        )

    prior = prior.float().to(device)

    return prior / (prior.sum() + 1e-8)

# =========================================================
# RESOLVE CF LEVELS
# =========================================================
def _resolve_cf_levels(cfg):

    levels = _cfg(cfg, 'cf_levels', None)

    if levels is None or len(levels) == 0:

        return DEFAULT_CF_LEVELS

    return levels

# =========================================================
# SAFE MODEL UNWRAP
# =========================================================
def get_base_model(model):

    return (
        model.module
        if isinstance(model, torch.nn.DataParallel)
        else model
    )

# =========================================================
# FORWARD WITH NODES
# =========================================================
def forward_with_nodes(model, x):

    out = model(x, return_nodes=True)

    if isinstance(out, tuple):

        if len(out) == 3:

            logits, nodes, ordinal_logits = out
            pooled = None

        elif len(out) == 4:

            logits, nodes, ordinal_logits, pooled = out

        else:

            logits = out[0]
            nodes = None
            ordinal_logits = None
            pooled = None

    else:

        logits = out
        nodes = None
        ordinal_logits = None
        pooled = None

    return logits, nodes, ordinal_logits, pooled

# =========================================================
# EXPECTED GRADE
# =========================================================
def expected_grade_from_probs(probs):

    grades = torch.arange(
        probs.size(-1),
        device=probs.device
    ).float()

    return (probs * grades).sum(dim=-1)

# =========================================================
# ORDINAL → CLASS PROBS
# =========================================================
def ordinal_logits_to_probs(ordinal_logits, num_classes=5):

    B      = ordinal_logits.size(0)
    device = ordinal_logits.device

    cum_probs = torch.sigmoid(ordinal_logits)

    ones  = torch.ones(B, 1, device=device)
    zeros = torch.zeros(B, 1, device=device)

    cum_full = torch.cat(
        [ones, cum_probs, zeros],
        dim=1
    )

    class_probs = (
        cum_full[:, :-1]
        - cum_full[:, 1:]
    )

    class_probs = torch.clamp(
        class_probs,
        min=1e-6
    )

    class_probs = class_probs / (
        class_probs.sum(dim=-1, keepdim=True)
        + 1e-8
    )

    return class_probs

# =========================================================
# ATTENTION SCORES
# =========================================================
def get_attention_scores(model, nodes):

    base = get_base_model(model)

    scores = base.attention_pool.attn(
        nodes
    ).squeeze(-1)

    return torch.softmax(scores, dim=1)

# =========================================================
# CLUSTER CONSISTENCY
# =========================================================
def compute_cluster_consistency(nodes):

    nodes_norm = F.normalize(nodes, dim=-1)

    sim = torch.bmm(
        nodes_norm,
        nodes_norm.transpose(1, 2)
    ).mean(dim=-1)

    sim_min = sim.min(
        dim=1,
        keepdim=True
    )[0]

    sim_max = sim.max(
        dim=1,
        keepdim=True
    )[0]

    return (
        (sim - sim_min)
        / (sim_max - sim_min + 1e-8)
    )

# =========================================================
# NODE IMPORTANCE
# =========================================================
def compute_node_scores(attn_scores, nodes):

    strength = torch.norm(nodes, dim=-1)

    strength = strength / (
        strength.max(dim=1, keepdim=True)[0]
        + 1e-8
    )

    cluster = compute_cluster_consistency(nodes)

    return (
        0.50 * attn_scores
        + 0.25 * strength
        + 0.25 * cluster
    )

# =========================================================
# BUILD CF NODES
# =========================================================
def build_counterfactual_nodes(
    nodes,
    node_scores,
    cf_levels=DEFAULT_CF_LEVELS
):

    if cf_levels is None or len(cf_levels) == 0:

        cf_levels = DEFAULT_CF_LEVELS

    B, N, D = nodes.shape

    idx = torch.argsort(
        node_scores,
        dim=1,
        descending=True
    )

    cf_nodes_all = []

    for ratio, suppression in cf_levels:

        k = max(int(N * ratio), 1)

        top_idx = idx[:, :k]

        cf_nodes = nodes.clone()

        scaling = torch.ones_like(cf_nodes)

        scaling.scatter_(
            1,
            top_idx.unsqueeze(-1).expand(-1, -1, D),
            suppression
        )

        cf_nodes_all.append(
            cf_nodes * scaling
        )

    return torch.cat(cf_nodes_all, dim=0), len(cf_levels)

# =========================================================
# CLASSIFY CF NODES
# =========================================================
def classify_cf_nodes(model, cf_nodes, global_feat):

    base = get_base_model(model)

    # =====================================================
    # NODE POOL
    # =====================================================
    pooled_nodes = base.attention_pool(cf_nodes)

    # =====================================================
    # REBUILD FUSED FEATURE
    # =====================================================
    fused = torch.cat(
        [
            pooled_nodes,
            global_feat
        ],
        dim=1
    )

    # =====================================================
    # FUSION BLOCK
    # =====================================================
    fused = base.fusion_block(fused)

    # =====================================================
    # DROPOUT
    # =====================================================
    fused = base.feature_dropout(fused)

    # =====================================================
    # CLASSIFIER
    # =====================================================
    logits = base.classifier(fused)

    return logits

# =========================================================
# STABLE CAUSAL DROP
# =========================================================
def compute_stable_causal_drop(exp_orig, exp_cf):

    drops = F.relu(
        exp_orig.unsqueeze(0) - exp_cf
    )

    mean_drop = drops.mean(dim=0)

    std_drop = drops.std(dim=0)

    return mean_drop / (std_drop + 0.15)

# =========================================================
# BLEND WEIGHT  (v1 formula — DO NOT CHANGE)
# =========================================================
def compute_blend_weight(stable_drop, confidence, cfg):

    sensitivity = _cfg(
        cfg,
        'sensitivity_threshold',
        0.08
    )

    cf_strength = _resolve_cf_strength(cfg)

    causal_gate = torch.sigmoid(
        5.0 * (stable_drop - sensitivity)
    )

    uncertainty = 1.0 - confidence

    alpha = cf_strength * (
        0.05 * causal_gate
        + 0.55 * causal_gate * uncertainty
    )

    return torch.clamp(alpha, 0.0, 0.65)

# =========================================================
# SOURCE PRIOR CORRECTION  (v1 formula — DO NOT CHANGE)
# =========================================================
def apply_source_prior_correction(probs, cfg):

    prior = _resolve_source_prior(
        cfg,
        probs.device
    )

    strength = _cfg(
        cfg,
        'prior_correction_strength',
        0.18
    )

    logits = torch.log(probs + 1e-8)

    logits = logits + (
        strength * torch.log(prior + 1e-8)
    )

    return F.softmax(logits, dim=-1)

# =========================================================
# SAFE NORMALIZE
# =========================================================
def safe_normalize(probs):

    probs = torch.nan_to_num(probs)

    probs = torch.clamp(
        probs,
        min=1e-6
    )

    probs = probs / (
        probs.sum(dim=-1, keepdim=True)
        + 1e-8
    )

    return probs

# =========================================================
# SINGLE VIEW
# =========================================================
def _cf_node_single_view(model, images, cfg):

    base_temp = _cfg(cfg, 'temp', 1.05)

    B = images.size(0)

    base = get_base_model(model)

    # =====================================================
    # ORIGINAL FORWARD
    # =====================================================
    logits_orig, nodes, ordinal_logits, _ = forward_with_nodes(
        model,
        images
    )

    probs_orig = F.softmax(
        logits_orig / base_temp,
        dim=-1
    )

    exp_orig = expected_grade_from_probs(
        probs_orig
    )

    confidence = probs_orig.max(
        dim=1
    ).values

    num_classes = probs_orig.size(-1)

    # =====================================================
    # FALLBACK
    # =====================================================
    if nodes is None:

        return probs_orig

    # =====================================================
    # RECOMPUTE GLOBAL FEATURES
    # =====================================================
    feat = base.features(images)

    global_feat = base.gem_pool(feat)

    global_feat = base.global_proj(global_feat)

    # =====================================================
    # NODE IMPORTANCE
    # =====================================================
    attn_scores = get_attention_scores(
        model,
        nodes
    )

    node_scores = compute_node_scores(
        attn_scores,
        nodes
    )

    # =====================================================
    # BUILD CF NODES
    # =====================================================
    cf_levels = _resolve_cf_levels(cfg)

    cf_nodes_all, num_cf = build_counterfactual_nodes(
        nodes,
        node_scores,
        cf_levels=cf_levels
    )

    # =====================================================
    # REPEAT GLOBAL FEATURES
    # =====================================================
    global_feat_repeat = global_feat.repeat(
        num_cf,
        1
    )

    # =====================================================
    # CF CLASSIFICATION
    # =====================================================
    logits_cf_all = classify_cf_nodes(
        model,
        cf_nodes_all,
        global_feat_repeat
    )

    probs_cf_all = F.softmax(
        logits_cf_all / base_temp,
        dim=-1
    )

    exp_cf = expected_grade_from_probs(
        probs_cf_all
    ).view(num_cf, B)

    # =====================================================
    # STABLE DROP
    # =====================================================
    stable_drop = compute_stable_causal_drop(
        exp_orig,
        exp_cf
    )

    # =====================================================
    # CAUSAL TEMPERATURE
    # =====================================================
    sample_temp = (
        base_temp
        - 0.22 * stable_drop
    )

    sample_temp = torch.clamp(
        sample_temp,
        min=0.72,
        max=1.08
    )

    # =====================================================
    # ORDINAL BLEND
    # =====================================================
    if ordinal_logits is not None:

        ordinal_probs = ordinal_logits_to_probs(
            ordinal_logits,
            num_classes=num_classes
        )

    else:

        ordinal_probs = probs_orig

    # =====================================================
    # BLEND
    # =====================================================
    alpha = compute_blend_weight(
        stable_drop,
        confidence,
        cfg
    ).unsqueeze(-1)

    blended_probs = (
        (1.0 - alpha) * probs_orig
        + alpha * ordinal_probs
    )

    # =====================================================
    # TEMPERATURE SHARPEN
    # =====================================================
    final_logits = (
        torch.log(blended_probs + 1e-8)
        / sample_temp.unsqueeze(-1)
    )

    return F.softmax(final_logits, dim=-1)

# =========================================================
# ADAPT BATCH  (argmax — v1 behaviour)
# =========================================================
@torch.no_grad()
def adapt_batch(model, images, cfg, return_probs=False):

    model.eval()

    view_probs_list = []

    for aug_fn in AUGMENTATIONS:

        view_probs = _cf_node_single_view(
            model,
            aug_fn(images),
            cfg
        )

        view_probs_list.append(view_probs)

    avg_probs = torch.stack(
        view_probs_list,
        dim=0
    ).mean(dim=0)

    # =====================================================
    # SOURCE PRIOR CORRECTION
    # =====================================================
    final_probs = apply_source_prior_correction(
        avg_probs,
        cfg
    )

    final_probs = safe_normalize(final_probs)

    # =====================================================
    # ARGMAX (v1)
    # =====================================================
    final_preds = torch.argmax(
        final_probs,
        dim=-1
    )

    # =====================================================
    # RANKING SCORE  (posterior expected grade)
    # =====================================================
    ranking_scores = expected_grade_from_probs(
        final_probs
    )

    preds_np = final_preds.cpu().numpy().astype(np.int64)

    probs_np = final_probs.cpu().numpy().astype(np.float32)

    ranking_np = ranking_scores.cpu().numpy().astype(np.float32)

    if return_probs:

        return (
            preds_np,
            probs_np,
            ranking_np
        )

    return preds_np

# =========================================================
# RUN TTA
# =========================================================
def run_tta(model, loader, cfg, return_probs=False):

    preds_all = []

    probs_all = []

    ranking_all = []

    labels_all = []

    device = next(model.parameters()).device

    for images, labels in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        if return_probs:

            preds, probs, ranking = adapt_batch(
                model,
                images,
                cfg,
                return_probs=True
            )

            probs_all.append(probs)

            ranking_all.append(ranking)

        else:

            preds = adapt_batch(
                model,
                images,
                cfg
            )

        preds_all.extend(preds)

        labels_all.extend(labels.numpy())

    preds_all = np.array(
        preds_all,
        dtype=np.int64
    )

    labels_all = np.array(
        labels_all,
        dtype=np.int64
    )

    if return_probs:

        probs_all = np.concatenate(
            probs_all,
            axis=0
        ).astype(np.float32)

        ranking_all = np.concatenate(
            ranking_all,
            axis=0
        ).astype(np.float32)

        return (
            preds_all,
            labels_all,
            probs_all,
            ranking_all
        )

    return preds_all, labels_all


### best so far It final best 

Writing /kaggle/working/tta_node.py


In [18]:
%%writefile  /kaggle/working/tta_config.py
# =========================================================
# /kaggle/working/tta_config_baselines.py
#
# Config ONLY for:
#   - TENT
#   - SAR
#   - EATA
#   - T3A
#
# Designed for:
#   EyePACS -> {MESSIDOR, IDRID, DeepDRiD, APTOS, DDR}
#
# Your setup:
#   - ResNet50 backbone
#   - DeepSets fusion
#   - LayerNorm heads
#   - 5-class DR grading
#
# IMPORTANT:
# We use CLASSIFIER-ONLY adaptation because:
#   - Fundus TTA is unstable with BN adaptation
#   - Your architecture is LN-heavy
#   - FUNOTTA observations align with this
# =========================================================

import math


class TTAConfig:

    # =====================================================
    # SHARED
    # =====================================================
    num_classes = 5

    batch_size = 32
    image_size = 512

    # =====================================================
    # TENT
    # =====================================================
    # Stable DR setting
    tent_mode = "classifier_only"

    # IMPORTANT:
    # 1e-3 is usually too aggressive for fundus
    tent_lr = 1e-4

    # Single-step adaptation is safest
    tent_steps = 1

    # Reset between target domains
    tent_episodic = True

    # Prevent exploding updates
    tent_grad_clip = 1.0

    # =====================================================
    # SAR
    # =====================================================
    # Much safer for DR than BN+LN adaptation
    sar_mode = "classifier_only"

    sar_lr = 1e-4

    # Canonical SAM perturbation
    sar_rho = 0.05

    sar_steps = 1

    # Paper default:
    # 0.4 * ln(C)
    sar_margin_e0 = 0.4 * math.log(num_classes)

    # EMA recovery threshold
    sar_reset_em = 0.2

    sar_grad_clip = 1.0

    # =====================================================
    # EATA
    # =====================================================
    # Stable DR setup
    eata_mode = "classifier_only"

    eata_lr = 1e-4

    eata_steps = 1

    # Reliable sample threshold
    eata_e_margin = 0.4 * math.log(num_classes)

    # Redundancy filtering
    eata_d_margin = 0.05

    # Fisher regularization OFF initially
    # (avoid complexity + instability)
    eata_use_fisher = False

    # Kept for completeness
    eata_fisher_alpha = 2000.0
    eata_fisher_steps = 200

    eata_grad_clip = 1.0

    # =====================================================
    # T3A
    # =====================================================
    # VERY important hyperparameter
    #
    # Recommended sweep:
    #   5, 20, 50
    #
    # 20 is strong default
    t3a_max_supports = 20

    # Temperature scaling
    t3a_temperature = 1.0

Overwriting /kaggle/working/tta_config.py


In [13]:
%%writefile /kaggle/working/tta_utils.py
#
# Shared utilities for TENT / SAR / EATA / T3A,
# tailored to your DeepSetModel (ResNet50 backbone +
# LayerNorm heads + ordinal head).
#
# Key fixes vs. previous version:
#   - configure_model splits into 3 explicit modes:
#       * bn_only         -> TENT / EATA canonical
#       * bn_and_ln       -> SAR canonical (skip layer4 + last LN)
#       * classifier_only -> "dagger" protocol (FunOTTA fundus)
#   - BN running stats are FROZEN (forces batch stats),
#     matching the official TENT/SAR/EATA repos.
#   - Adds save/load of model+optimizer state for episodic
#     resets (per-domain reset is critical for fair eval).
#   - T3A feature extractor returns 64-d, the *true* input
#     to the final classifier linear, so cosine-similarity
#     prototypes match the trained classifier geometry.
# =========================================================

import copy
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


# -------------------- model unwrap --------------------

def get_base_model(model):
    return model.module if isinstance(model, torch.nn.DataParallel) else model


# -------------------- forward helpers --------------------

def forward_logits(model, x):
    """Robust forward: handles (logits), (logits, nodes), (logits, nodes, ord)."""
    out = model(x)
    if isinstance(out, tuple):
        return out[0]
    return out


def _features_fused(base, x):
    """Return the 256-d fused feature (post FusionBlock)."""
    feat = base.features(x)
    global_feat = base.global_proj(base.gem_pool(feat))
    nodes = feat.flatten(2).transpose(1, 2)
    nodes = base.node_proj(nodes)
    pooled_nodes = base.attention_pool(nodes)
    fused = torch.cat([pooled_nodes, global_feat], dim=1)
    fused = base.fusion_block(fused)
    return fused


def extract_features(model, x):
    """Backwards-compatible alias: returns 256-d fused feature."""
    return _features_fused(get_base_model(model), x)


def extract_t3a_features(model, x):
    """
    Features that feed the FINAL classifier linear (64-d).
    Required for canonical T3A so that prototype cosine
    similarity matches the trained classifier geometry.
    """
    base = get_base_model(model)
    fused = _features_fused(base, x)
    h = fused
    # apply every layer of the classifier head EXCEPT the last Linear
    for layer in list(base.classifier.children())[:-1]:
        h = layer(h)
    return h  # shape [B, 64]


def get_final_linear(model):
    """Return the final nn.Linear of the classifier head (the prototype layer)."""
    base = get_base_model(model)
    last = list(base.classifier.children())[-1]
    assert isinstance(last, nn.Linear), \
        "Final layer of classifier must be nn.Linear for T3A."
    return last


# -------------------- entropy --------------------

@torch.jit.script
def softmax_entropy(x: torch.Tensor) -> torch.Tensor:
    """Entropy of softmax(logits) along dim=1."""
    return -(x.softmax(1) * x.log_softmax(1)).sum(1)


# -------------------- save / restore --------------------

def copy_model_and_optimizer(model, optimizer):
    return (
        copy.deepcopy(model.state_dict()),
        copy.deepcopy(optimizer.state_dict()) if optimizer is not None else None,
    )


def load_model_and_optimizer(model, optimizer, model_state, optimizer_state):
    model.load_state_dict(model_state, strict=True)
    if optimizer is not None and optimizer_state is not None:
        optimizer.load_state_dict(optimizer_state)


# -------------------- BN handling --------------------

def freeze_bn_running_stats(model):
    """
    Force BN to use BATCH statistics during TTA.
    This is what the TENT/SAR/EATA papers do; without it,
    you are still using stale source-domain running stats
    and adaptation barely helps.
    """
    base = get_base_model(model)
    for m in base.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.track_running_stats = False
            m.running_mean = None
            m.running_var = None
    return model


# -------------------- configure modes --------------------

def configure_model_bn_only(model):
    """
    Canonical TENT / EATA setup:
      - model.train()
      - all params frozen
      - BN affine (gamma, beta) require_grad
      - BN running stats disabled (batch stats only)
    """
    base = get_base_model(model)
    base.train()
    base.requires_grad_(False)
    for m in base.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.requires_grad_(True)
            m.track_running_stats = False
            m.running_mean = None
            m.running_var = None
    return model


def configure_model_bn_and_ln(model, skip_last_block=True, skip_classifier_ln=True):
    """
    Canonical SAR setup:
      - update BN in backbone (except layer4 if skip_last_block)
      - update LN in heads (except inside classifier head if
        skip_classifier_ln; this LN is right before the prediction)
      - BN running stats disabled (batch stats only)
    """
    base = get_base_model(model)
    base.train()
    base.requires_grad_(False)

    for nm, m in base.named_modules():
        if skip_last_block and nm.startswith("features.7"):
            # ResNet50 layer4 lives at features.7 in your nn.Sequential wrap
            continue
        if skip_classifier_ln and nm.startswith("classifier"):
            # don't touch the LN feeding the final logits
            continue

        if isinstance(m, nn.BatchNorm2d):
            m.requires_grad_(True)
            m.track_running_stats = False
            m.running_mean = None
            m.running_var = None
        elif isinstance(m, (nn.LayerNorm, nn.GroupNorm)):
            m.requires_grad_(True)
    return model


def configure_model_classifier_only(model):
    """
    "Dagger" protocol from FunOTTA: only the classifier head
    (and ordinal head) get adapted. This is what FunOTTA found
    works on fundus when full BN-style adaptation collapses.
    """
    base = get_base_model(model)
    base.train()
    base.requires_grad_(False)
    for p in base.classifier.parameters():
        p.requires_grad_(True)
    if hasattr(base, "ordinal_head"):
        for p in base.ordinal_head.parameters():
            p.requires_grad_(True)
    # backbone stays in eval mode so BN uses source running stats
    base.features.eval()
    return model


# -------------------- collect params --------------------

def collect_bn_params(model, skip_last_block=False):
    base = get_base_model(model)
    params, names = [], []
    for nm, m in base.named_modules():
        if skip_last_block and nm.startswith("features.7"):
            continue
        if isinstance(m, nn.BatchNorm2d):
            for pn, p in m.named_parameters(recurse=False):
                if pn in ("weight", "bias") and p.requires_grad:
                    params.append(p)
                    names.append(f"{nm}.{pn}")
    return params, names


def collect_ln_params(model, skip_classifier_ln=True):
    base = get_base_model(model)
    params, names = [], []
    for nm, m in base.named_modules():
        if skip_classifier_ln and nm.startswith("classifier"):
            continue
        if isinstance(m, (nn.LayerNorm, nn.GroupNorm)):
            for pn, p in m.named_parameters(recurse=False):
                if pn in ("weight", "bias") and p.requires_grad:
                    params.append(p)
                    names.append(f"{nm}.{pn}")
    return params, names


def collect_classifier_params(model):
    base = get_base_model(model)
    params, names = [], []
    for nm, p in base.classifier.named_parameters():
        if p.requires_grad:
            params.append(p)
            names.append(f"classifier.{nm}")
    if hasattr(base, "ordinal_head"):
        for nm, p in base.ordinal_head.named_parameters():
            if p.requires_grad:
                params.append(p)
                names.append(f"ordinal_head.{nm}")
    return params, names


def collect_params(model, mode="bn_only"):
    """
    Unified entry point.
      mode in {"bn_only", "bn_and_ln_sar", "classifier_only"}
    """
    if mode == "bn_only":
        return collect_bn_params(model, skip_last_block=False)
    if mode == "bn_and_ln_sar":
        bn_p, bn_n = collect_bn_params(model, skip_last_block=True)
        ln_p, ln_n = collect_ln_params(model, skip_classifier_ln=True)
        return bn_p + ln_p, bn_n + ln_n
    if mode == "classifier_only":
        return collect_classifier_params(model)
    raise ValueError(f"Unknown mode: {mode}")


# -------------------- defaults helpers --------------------

def default_e_margin(num_classes, scale=0.4):
    """E_0 = scale * ln(C). With scale=0.4 this matches the SAR paper."""
    return scale * math.log(num_classes)



# =========================================================
# T3A HELPERS
# =========================================================

import torch
import torch.nn as nn


def get_final_linear(model):
    """
    Returns final linear classifier layer.

    Supports:
        - plain nn.Linear
        - model.fc
        - model.classifier
        - nested modules
    """

    # Common naming
    if hasattr(model, "fc") and isinstance(model.fc, nn.Linear):
        return model.fc

    if hasattr(model, "classifier") and isinstance(model.classifier, nn.Linear):
        return model.classifier

    # Search recursively
    linear_layers = []

    for m in model.modules():
        if isinstance(m, nn.Linear):
            linear_layers.append(m)

    if len(linear_layers) == 0:
        raise RuntimeError(
            "No linear layer found in model."
        )

    # final linear
    return linear_layers[-1]


@torch.no_grad()
def extract_features(model, x):
    """
    Extract penultimate features for T3A.

    Assumes:
        final layer is Linear
    """

    final_linear = get_final_linear(model)

    features = None

    def hook_fn(module, input, output):
        nonlocal features
        features = input[0]

    handle = final_linear.register_forward_hook(hook_fn)

    _ = model(x)

    handle.remove()

    if features is None:
        raise RuntimeError(
            "Failed to extract features."
        )

    return features

Writing /kaggle/working/tta_utils.py


In [14]:
%%writefile /kaggle/working/tta_tent.py

import copy
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F


# =========================================================
# ENTROPY
# =========================================================
def softmax_entropy(x):

    return -(x.softmax(1) * x.log_softmax(1)).sum(1)


# =========================================================
# EXTRACT LOGITS
# =========================================================
def extract_logits(output):

    # tuple/list output
    if isinstance(output, (tuple, list)):

        return output[0]

    # dict output
    if isinstance(output, dict):

        if "logits" in output:
            return output["logits"]

        first_key = list(output.keys())[0]

        return output[first_key]

    # tensor output
    return output


# =========================================================
# COLLECT PARAMS
# =========================================================
def collect_params(

    model,
    mode="classifier_only"
):

    params = []

    names = []

    if mode == "classifier_only":

        for name, m in model.named_modules():

            if isinstance(m, nn.Linear):

                if m.weight.requires_grad:

                    params.append(m.weight)

                    names.append(
                        f"{name}.weight"
                    )

                if (

                    m.bias is not None
                    and m.bias.requires_grad
                ):

                    params.append(m.bias)

                    names.append(
                        f"{name}.bias"
                    )

    else:

        raise ValueError(
            f"Unsupported mode: {mode}"
        )

    return params, names


# =========================================================
# CONFIGURE MODEL
# =========================================================
def configure_model(model):

    model.eval()

    model.requires_grad_(False)

    for m in model.modules():

        if isinstance(m, nn.Linear):

            m.requires_grad_(True)

    return model


# =========================================================
# TENT
# =========================================================
class Tent(nn.Module):

    def __init__(

        self,

        model,

        optimizer,

        steps=1,

        episodic=True,

        grad_clip=1.0,
    ):
        super().__init__()

        self.model = model

        self.optimizer = optimizer

        self.steps = steps

        self.episodic = episodic

        self.grad_clip = grad_clip

        # restore states
        self.model_state = copy.deepcopy(
            model.state_dict()
        )

        self.optimizer_state = copy.deepcopy(
            optimizer.state_dict()
        )

    # =====================================================
    # RESET
    # =====================================================
    def reset(self):

        self.model.load_state_dict(
            self.model_state,
            strict=True
        )

        self.optimizer.load_state_dict(
            self.optimizer_state
        )

    # =====================================================
    # ADAPT
    # =====================================================
    @torch.enable_grad()
    def forward_and_adapt(self, x):

        output = self.model(x)

        logits = extract_logits(output)

        loss = softmax_entropy(
            logits
        ).mean()

        self.optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(

            self.model.parameters(),

            self.grad_clip
        )

        self.optimizer.step()

        return logits

    # =====================================================
    # FORWARD
    # =====================================================
    def forward(self, x):

        if self.episodic:

            self.reset()

        logits = None

        for _ in range(self.steps):

            logits = self.forward_and_adapt(x)

        return logits


# =========================================================
# RUN
# =========================================================
def run_tent(

    model,

    loader,

    cfg,

    return_probs=False
):

    mode = getattr(
        cfg,
        "tent_mode",
        "classifier_only"
    )

    steps = getattr(
        cfg,
        "tent_steps",
        1
    )

    episodic = getattr(
        cfg,
        "tent_episodic",
        True
    )

    lr = getattr(
        cfg,
        "tent_lr",
        1e-4
    )

    grad_clip = getattr(
        cfg,
        "tent_grad_clip",
        1.0
    )

    # =====================================================
    # CONFIGURE
    # =====================================================
    model = configure_model(model)

    params, _ = collect_params(

        model,

        mode=mode
    )

    optimizer = torch.optim.Adam(

        params,

        lr=lr
    )

    tent_model = Tent(

        model=model,

        optimizer=optimizer,

        steps=steps,

        episodic=episodic,

        grad_clip=grad_clip,
    )

    # =====================================================
    # INFERENCE
    # =====================================================
    preds_all = []

    probs_all = []

    labels_all = []

    device = next(
        model.parameters()
    ).device

    for images, labels in loader:

        images = images.to(

            device,

            non_blocking=True
        )

        logits = tent_model(images)

        probs = F.softmax(

            logits,

            dim=1
        )

        preds = probs.argmax(dim=1)

        preds_all.extend(
            preds.detach().cpu().numpy()
        )

        probs_all.append(
            probs.detach().cpu().numpy()
        )

        labels_all.extend(
            labels.numpy()
        )

    preds_all = np.array(preds_all)

    labels_all = np.array(labels_all)

    probs_all = np.concatenate(
        probs_all,
        axis=0
    )

    if return_probs:

        return (

            preds_all,

            labels_all,

            probs_all
        )

    return preds_all, labels_all


# =========================================================
# BACKWARD COMPATIBILITY
# =========================================================
run_tta = run_tent

Writing /kaggle/working/tta_tent.py


In [15]:
%%writefile /kaggle/working/tta_t3a.py

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F


# =========================================================
# SAFE LOGIT EXTRACTION
# =========================================================
def extract_logits(output):

    if isinstance(output, (tuple, list)):
        return output[0]

    if isinstance(output, dict):

        if "logits" in output:
            return output["logits"]

        return list(output.values())[0]

    return output


# =========================================================
# ENTROPY
# =========================================================
def softmax_entropy(x):

    return -(x.softmax(1) * x.log_softmax(1)).sum(1)


# =========================================================
# SIMPLE T3A
# =========================================================
class T3A(nn.Module):

    def __init__(

        self,

        model,

        num_classes=5,

        filter_K=20
    ):
        super().__init__()

        self.model = model

        self.num_classes = num_classes

        self.filter_K = filter_K

        self.supports = None

        self.labels = None

        self.ent = None

        self.model.eval()

        for p in self.model.parameters():

            p.requires_grad_(False)

    # =====================================================
    # FORWARD
    # =====================================================
    @torch.no_grad()
    def forward(self, x):

        output = self.model(x)

        logits = extract_logits(output)

        # -------------------------------------------------
        # USE LOGITS AS FEATURES
        # avoids all architecture mismatch
        # -------------------------------------------------
        z = F.normalize(
            logits,
            dim=1
        )

        yhat = F.one_hot(

            logits.argmax(1),

            num_classes=self.num_classes

        ).float()

        ent = softmax_entropy(logits)

        # -------------------------------------------------
        # INIT
        # -------------------------------------------------
        if self.supports is None:

            self.supports = z.clone()

            self.labels = yhat.clone()

            self.ent = ent.clone()

        else:

            self.supports = torch.cat(
                [self.supports, z],
                dim=0
            )

            self.labels = torch.cat(
                [self.labels, yhat],
                dim=0
            )

            self.ent = torch.cat(
                [self.ent, ent],
                dim=0
            )

        # -------------------------------------------------
        # FILTER
        # -------------------------------------------------
        supports_new = []

        labels_new = []

        ent_new = []

        for c in range(self.num_classes):

            mask = self.labels[:, c] > 0

            if mask.sum() == 0:
                continue

            cls_sup = self.supports[mask]

            cls_lab = self.labels[mask]

            cls_ent = self.ent[mask]

            if (

                self.filter_K > 0

                and

                cls_ent.numel() > self.filter_K
            ):

                idx = torch.argsort(
                    cls_ent
                )[:self.filter_K]

                cls_sup = cls_sup[idx]

                cls_lab = cls_lab[idx]

                cls_ent = cls_ent[idx]

            supports_new.append(cls_sup)

            labels_new.append(cls_lab)

            ent_new.append(cls_ent)

        self.supports = torch.cat(
            supports_new,
            dim=0
        )

        self.labels = torch.cat(
            labels_new,
            dim=0
        )

        self.ent = torch.cat(
            ent_new,
            dim=0
        )

        # -------------------------------------------------
        # PROTOTYPES
        # -------------------------------------------------
        protos = []

        for c in range(self.num_classes):

            mask = self.labels[:, c] > 0

            if mask.sum() == 0:

                proto = torch.zeros(

                    self.supports.shape[1],

                    device=self.supports.device
                )

            else:

                proto = self.supports[
                    mask
                ].mean(0)

            proto = F.normalize(

                proto.unsqueeze(0),

                dim=1

            ).squeeze(0)

            protos.append(proto)

        protos = torch.stack(
            protos,
            dim=0
        )

        logits = z @ protos.t()

        return logits


# =========================================================
# RUN
# =========================================================
def run_t3a(

    model,

    loader,

    cfg,

    return_probs=False
):

    num_classes = getattr(
        cfg,
        "num_classes",
        5
    )

    filter_K = getattr(
        cfg,
        "t3a_max_supports",
        20
    )

    temperature = getattr(
        cfg,
        "t3a_temperature",
        1.0
    )

    t3a = T3A(

        model,

        num_classes=num_classes,

        filter_K=filter_K
    )

    preds_all = []

    probs_all = []

    labels_all = []

    device = next(
        model.parameters()
    ).device

    for images, labels in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        logits = t3a(images)

        probs = F.softmax(
            logits / temperature,
            dim=1
        )

        preds = probs.argmax(1)

        preds_all.extend(
            preds.detach().cpu().numpy()
        )

        probs_all.append(
            probs.detach().cpu().numpy()
        )

        labels_all.extend(
            labels.numpy()
        )

    preds_all = np.array(preds_all)

    labels_all = np.array(labels_all)

    probs_all = np.concatenate(
        probs_all,
        axis=0
    )

    if return_probs:

        return (
            preds_all,
            labels_all,
            probs_all
        )

    return preds_all, labels_all

Writing /kaggle/working/tta_t3a.py


In [16]:
%%writefile /kaggle/working/tta_eata.py

# =========================================================
# Stable Canonical EATA for DR Grading
#
# Fixed for:
#   - tuple-output models
#   - dict-output models
#   - DeepSets / CF-node architectures
#   - classifier-only adaptation
#   - multi-GPU models
#   - Final_Run.py compatibility
#
# Paper:
# Efficient Test-Time Model Adaptation without
# Forgetting (ICML 2022)
# =========================================================

import copy
import math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F


# =========================================================
# ENTROPY
# =========================================================
def softmax_entropy(x):

    return -(x.softmax(1) * x.log_softmax(1)).sum(1)


# =========================================================
# SAFE LOGIT EXTRACTION
# =========================================================
def extract_logits(output):

    # tuple/list output
    if isinstance(output, (tuple, list)):

        return output[0]

    # dict output
    if isinstance(output, dict):

        if "logits" in output:

            return output["logits"]

        first_key = list(output.keys())[0]

        return output[first_key]

    # tensor output
    return output


# =========================================================
# FORWARD LOGITS
# =========================================================
def forward_logits(model, x):

    output = model(x)

    return extract_logits(output)


# =========================================================
# DEFAULT E-MARGIN
# =========================================================
def default_e_margin(

    num_classes=5,

    scale=0.4
):

    return scale * math.log(num_classes)


# =========================================================
# CONFIGURE MODEL
# =========================================================
def configure_model_classifier_only(model):

    model.eval()

    model.requires_grad_(False)

    for m in model.modules():

        if isinstance(m, nn.Linear):

            m.requires_grad_(True)

    return model


# =========================================================
# COLLECT PARAMS
# =========================================================
def collect_params(

    model,

    mode="classifier_only"
):

    params = []

    names = []

    if mode != "classifier_only":

        raise ValueError(
            f"Unsupported mode: {mode}"
        )

    for name, m in model.named_modules():

        if isinstance(m, nn.Linear):

            if m.weight.requires_grad:

                params.append(m.weight)

                names.append(
                    f"{name}.weight"
                )

            if (

                m.bias is not None

                and

                m.bias.requires_grad
            ):

                params.append(m.bias)

                names.append(
                    f"{name}.bias"
                )

    return params, names


# =========================================================
# COPY STATES
# =========================================================
def copy_model_and_optimizer(

    model,

    optimizer
):

    model_state = copy.deepcopy(
        model.state_dict()
    )

    optimizer_state = copy.deepcopy(
        optimizer.state_dict()
    )

    return model_state, optimizer_state


# =========================================================
# LOAD STATES
# =========================================================
def load_model_and_optimizer(

    model,

    optimizer,

    model_state,

    optimizer_state
):

    model.load_state_dict(

        model_state,

        strict=True
    )

    optimizer.load_state_dict(
        optimizer_state
    )


# =========================================================
# PROBABILITY EMA
# =========================================================
def update_probs(

    current,

    new_probs,

    momentum=0.9
):

    if current is None:

        if new_probs.numel() == 0:

            return None

        return new_probs.mean(0).detach()

    if new_probs.numel() == 0:

        return current

    with torch.no_grad():

        return (

            momentum * current

            +

            (1 - momentum)
            * new_probs.mean(0)
        )


# =========================================================
# COMPUTE FISHERS
# =========================================================
def compute_fishers(

    model,

    source_loader,

    num_classes,

    device,

    mode="classifier_only",

    num_steps=200
):

    model = configure_model_classifier_only(
        model
    )

    params, names = collect_params(

        model,

        mode=mode
    )

    theta_src = {

        n: p.detach().clone()

        for n, p in zip(names, params)
    }

    fishers = {

        n: torch.zeros_like(p)

        for n, p in zip(names, params)
    }

    ce = nn.CrossEntropyLoss()

    model.train()

    seen = 0

    for step, (x, y) in enumerate(source_loader):

        if step >= num_steps:

            break

        x = x.to(

            device,

            non_blocking=True
        )

        y = y.to(

            device,

            non_blocking=True
        )

        for p in params:

            if p.grad is not None:

                p.grad.zero_()

        logits = forward_logits(
            model,
            x
        )

        loss = ce(logits, y)

        loss.backward()

        for n, p in zip(names, params):

            if p.grad is not None:

                fishers[n] += (

                    p.grad.detach() ** 2
                )

        seen += 1

    if seen > 0:

        for n in fishers:

            fishers[n] /= seen

    return {

        n: (

            fishers[n],

            theta_src[n]
        )

        for n in fishers
    }


# =========================================================
# EATA MODULE
# =========================================================
class EATA(nn.Module):

    def __init__(

        self,

        model,

        optimizer,

        params_for_ewc=None,

        param_names_for_ewc=None,

        fishers=None,

        fisher_alpha=2000.0,

        steps=1,

        e_margin=None,

        d_margin=0.05,

        grad_clip=1.0,

        num_classes=5,
    ):
        super().__init__()

        self.model = model

        self.optimizer = optimizer

        self.steps = steps

        self.e_margin = (

            e_margin

            if e_margin is not None

            else default_e_margin(
                num_classes,
                0.4
            )
        )

        self.d_margin = d_margin

        self.grad_clip = grad_clip

        self.fishers = fishers

        self.fisher_alpha = fisher_alpha

        self.params_for_ewc = (
            params_for_ewc or []
        )

        self.param_names_for_ewc = (
            param_names_for_ewc or []
        )

        self.current_model_probs = None

        (
            self.model_state,

            self.optimizer_state

        ) = copy_model_and_optimizer(

            self.model,

            self.optimizer
        )

    # =====================================================
    # RESET
    # =====================================================
    def reset(self):

        load_model_and_optimizer(

            self.model,

            self.optimizer,

            self.model_state,

            self.optimizer_state
        )

        self.current_model_probs = None

    # =====================================================
    # FORWARD
    # =====================================================
    def forward(self, x):

        outputs = None

        for _ in range(self.steps):

            outputs = self.forward_and_adapt(
                x
            )

        return outputs

    # =====================================================
    # FORWARD + ADAPT
    # =====================================================
    @torch.enable_grad()
    def forward_and_adapt(self, x):

        self.optimizer.zero_grad(
            set_to_none=True
        )

        outputs = forward_logits(
            self.model,
            x
        )

        probs = outputs.softmax(1)

        # -------------------------------------------------
        # FILTER 1
        # RELIABLE SAMPLES
        # -------------------------------------------------
        ent = softmax_entropy(outputs)

        keep1 = torch.where(
            ent < self.e_margin
        )

        ent_kept = ent[keep1]

        probs_kept = probs[keep1]

        if ent_kept.numel() == 0:

            return outputs.detach()

        # -------------------------------------------------
        # FILTER 2
        # NON-REDUNDANT
        # -------------------------------------------------
        if self.current_model_probs is not None:

            cos_sim = F.cosine_similarity(

                self.current_model_probs.unsqueeze(0),

                probs_kept,

                dim=1
            )

            keep2 = torch.where(

                cos_sim.abs()
                <
                self.d_margin
            )

            ent_final = ent_kept[keep2]

            probs_final = probs_kept[keep2]

        else:

            ent_final = ent_kept

            probs_final = probs_kept

        # -------------------------------------------------
        # UPDATE EMA
        # -------------------------------------------------
        self.current_model_probs = update_probs(

            self.current_model_probs,

            probs_final,

            momentum=0.9
        )

        if ent_final.numel() == 0:

            return outputs.detach()

        # -------------------------------------------------
        # ENTROPY REWEIGHTING
        # -------------------------------------------------
        coeff = 1.0 / torch.exp(

            ent_final.detach()
            -
            self.e_margin
        )

        loss = (

            ent_final * coeff

        ).mean()

        # -------------------------------------------------
        # OPTIONAL FISHER REGULARIZATION
        # -------------------------------------------------
        if (

            self.fishers is not None

            and

            len(self.params_for_ewc) > 0
        ):

            ewc = 0.0

            for nm, p in zip(

                self.param_names_for_ewc,

                self.params_for_ewc
            ):

                if nm in self.fishers:

                    f_diag, theta_src = (
                        self.fishers[nm]
                    )

                    ewc = ewc + (

                        f_diag
                        *
                        (p - theta_src) ** 2

                    ).sum()

            loss = (

                loss

                +

                self.fisher_alpha * ewc
            )

        # -------------------------------------------------
        # UPDATE
        # -------------------------------------------------
        loss.backward()

        if self.grad_clip is not None:

            torch.nn.utils.clip_grad_norm_(

                [

                    p

                    for g in self.optimizer.param_groups

                    for p in g["params"]
                ],

                self.grad_clip
            )

        self.optimizer.step()

        return outputs.detach()


# =========================================================
# RUN EATA
# =========================================================
def run_eata(

    model,

    loader,

    cfg,

    return_probs=False,

    source_loader=None,

    fisher_loader=None,   # compatibility fix
):

    # -----------------------------------------------------
    # compatibility
    # -----------------------------------------------------
    if source_loader is None:

        source_loader = fisher_loader

    num_classes = getattr(
        cfg,
        "num_classes",
        5
    )

    mode = getattr(
        cfg,
        "eata_mode",
        "classifier_only"
    )

    lr = getattr(
        cfg,
        "eata_lr",
        1e-4
    )

    steps = getattr(
        cfg,
        "eata_steps",
        1
    )

    e_margin = getattr(

        cfg,

        "eata_e_margin",

        default_e_margin(
            num_classes,
            0.4
        )
    )

    d_margin = getattr(
        cfg,
        "eata_d_margin",
        0.05
    )

    fisher_alpha = getattr(
        cfg,
        "eata_fisher_alpha",
        2000.0
    )

    use_fisher = getattr(
        cfg,
        "eata_use_fisher",
        False
    )

    fisher_steps = getattr(
        cfg,
        "eata_fisher_steps",
        200
    )

    grad_clip = getattr(
        cfg,
        "eata_grad_clip",
        1.0
    )

    device = next(
        model.parameters()
    ).device

    # =====================================================
    # OPTIONAL FISHERS
    # =====================================================
    fishers = None

    if use_fisher:

        if source_loader is None:

            raise ValueError(
                "EATA Fisher requires source_loader."
            )

        fishers = compute_fishers(

            model,

            source_loader,

            num_classes,

            device,

            mode=mode,

            num_steps=fisher_steps,
        )

    # =====================================================
    # CONFIGURE
    # =====================================================
    model = configure_model_classifier_only(
        model
    )

    params, names = collect_params(

        model,

        mode=mode
    )

    optimizer = torch.optim.Adam(

        params,

        lr=lr,

        betas=(0.9, 0.999)
    )

    eata = EATA(

        model,

        optimizer,

        params_for_ewc=params,

        param_names_for_ewc=names,

        fishers=fishers,

        fisher_alpha=fisher_alpha,

        steps=steps,

        e_margin=e_margin,

        d_margin=d_margin,

        grad_clip=grad_clip,

        num_classes=num_classes,
    )

    eata.reset()

    # =====================================================
    # INFERENCE
    # =====================================================
    preds_all = []

    probs_all = []

    labels_all = []

    for images, labels in loader:

        images = images.to(

            device,

            non_blocking=True
        )

        logits = eata(images)

        probs = F.softmax(

            logits,

            dim=1
        )

        preds = probs.argmax(dim=1)

        preds_all.extend(
            preds.detach().cpu().numpy()
        )

        probs_all.append(
            probs.detach().cpu().numpy()
        )

        labels_all.extend(
            labels.numpy()
        )

    preds_all = np.array(preds_all)

    labels_all = np.array(labels_all)

    probs_all = np.concatenate(
        probs_all,
        axis=0
    )

    if return_probs:

        return (

            preds_all,

            labels_all,

            probs_all
        )

    return preds_all, labels_all

Writing /kaggle/working/tta_eata.py


In [17]:
%%writefile /kaggle/working/tta_sar.py

import copy
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F


# =========================================================
# ENTROPY
# =========================================================
def softmax_entropy(x):

    return -(x.softmax(1) * x.log_softmax(1)).sum(1)


# =========================================================
# EXTRACT LOGITS
# =========================================================
def extract_logits(output):

    # tuple/list
    if isinstance(output, (tuple, list)):

        return output[0]

    # dict
    if isinstance(output, dict):

        if "logits" in output:
            return output["logits"]

        first_key = list(output.keys())[0]

        return output[first_key]

    # tensor
    return output


# =========================================================
# CONFIGURE MODEL
# =========================================================
def configure_model(model):

    model.eval()

    model.requires_grad_(False)

    # classifier-only adaptation
    for m in model.modules():

        if isinstance(m, nn.Linear):

            m.requires_grad_(True)

    return model


# =========================================================
# COLLECT PARAMS
# =========================================================
def collect_params(

    model,
    mode="classifier_only"
):

    params = []

    names = []

    if mode == "classifier_only":

        for name, m in model.named_modules():

            if isinstance(m, nn.Linear):

                if m.weight.requires_grad:

                    params.append(m.weight)

                    names.append(
                        f"{name}.weight"
                    )

                if (

                    m.bias is not None
                    and m.bias.requires_grad
                ):

                    params.append(m.bias)

                    names.append(
                        f"{name}.bias"
                    )

    else:

        raise ValueError(
            f"Unsupported SAR mode: {mode}"
        )

    return params, names


# =========================================================
# SAM OPTIMIZER
# =========================================================
class SAM(torch.optim.Optimizer):

    def __init__(

        self,

        params,

        base_optimizer,

        rho=0.05,

        **kwargs
    ):

        assert rho >= 0.0

        defaults = dict(
            rho=rho,
            **kwargs
        )

        super().__init__(
            params,
            defaults
        )

        self.base_optimizer = base_optimizer(
            self.param_groups,
            **kwargs
        )

        self.param_groups = self.base_optimizer.param_groups

    @torch.no_grad()
    def first_step(

        self,

        zero_grad=False
    ):

        grad_norm = self._grad_norm()

        for group in self.param_groups:

            scale = group["rho"] / (
                grad_norm + 1e-12
            )

            for p in group["params"]:

                if p.grad is None:
                    continue

                e_w = p.grad * scale.to(p)

                p.add_(e_w)

                self.state[p]["e_w"] = e_w

        if zero_grad:
            self.zero_grad()

    @torch.no_grad()
    def second_step(

        self,

        zero_grad=False
    ):

        for group in self.param_groups:

            for p in group["params"]:

                if p.grad is None:
                    continue

                p.sub_(
                    self.state[p]["e_w"]
                )

        self.base_optimizer.step()

        if zero_grad:
            self.zero_grad()

    @torch.no_grad()
    def step(self):

        raise NotImplementedError

    @torch.no_grad()
    def _grad_norm(self):

        shared_device = self.param_groups[0][
            "params"
        ][0].device

        norm = torch.norm(

            torch.stack([

                p.grad.norm(p=2).to(shared_device)

                for group in self.param_groups

                for p in group["params"]

                if p.grad is not None
            ]),

            p=2
        )

        return norm


# =========================================================
# SAR
# =========================================================
class SAR(nn.Module):

    def __init__(

        self,

        model,

        optimizer,

        margin_e0,

        reset_em,

        steps=1,

        episodic=True,

        grad_clip=1.0,
    ):
        super().__init__()

        self.model = model

        self.optimizer = optimizer

        self.margin_e0 = margin_e0

        self.reset_em = reset_em

        self.steps = steps

        self.episodic = episodic

        self.grad_clip = grad_clip

        self.model_state = copy.deepcopy(
            model.state_dict()
        )

        self.optimizer_state = copy.deepcopy(
            optimizer.state_dict()
        )

        self.ema = None

    # =====================================================
    # RESET
    # =====================================================
    def reset(self):

        self.model.load_state_dict(
            self.model_state,
            strict=True
        )

        self.optimizer.load_state_dict(
            self.optimizer_state
        )

        self.ema = None

    # =====================================================
    # FORWARD + ADAPT
    # =====================================================
    @torch.enable_grad()
    def forward_and_adapt(self, x):

        # --------------------------------------------
        # FIRST FORWARD
        # --------------------------------------------
        output = self.model(x)

        logits = extract_logits(output)

        entropies = softmax_entropy(logits)

        filter_ids_1 = entropies < self.margin_e0

        if filter_ids_1.sum() == 0:

            return logits

        loss = entropies[
            filter_ids_1
        ].mean()

        self.optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(

            self.model.parameters(),

            self.grad_clip
        )

        self.optimizer.first_step(
            zero_grad=True
        )

        # --------------------------------------------
        # SECOND FORWARD
        # --------------------------------------------
        output_second = self.model(x)

        logits_second = extract_logits(
            output_second
        )

        entropies_second = softmax_entropy(
            logits_second
        )

        loss_second = entropies_second[
            filter_ids_1
        ].mean()

        loss_second.backward()

        torch.nn.utils.clip_grad_norm_(

            self.model.parameters(),

            self.grad_clip
        )

        self.optimizer.second_step(
            zero_grad=True
        )

        # --------------------------------------------
        # EMA RECOVERY
        # --------------------------------------------
        loss_value = loss_second.item()

        if self.ema is None:

            self.ema = loss_value

        else:

            self.ema = (
                0.9 * self.ema
                +
                0.1 * loss_value
            )

        if self.ema < self.reset_em:

            self.reset()

        return logits_second

    # =====================================================
    # FORWARD
    # =====================================================
    def forward(self, x):

        if self.episodic:

            self.reset()

        logits = None

        for _ in range(self.steps):

            logits = self.forward_and_adapt(x)

        return logits


# =========================================================
# RUN SAR
# =========================================================
def run_sar(

    model,

    loader,

    cfg,

    return_probs=False
):

    mode = getattr(
        cfg,
        "sar_mode",
        "classifier_only"
    )

    lr = getattr(
        cfg,
        "sar_lr",
        1e-4
    )

    rho = getattr(
        cfg,
        "sar_rho",
        0.05
    )

    steps = getattr(
        cfg,
        "sar_steps",
        1
    )

    margin_e0 = getattr(
        cfg,
        "sar_margin_e0",
        0.4 * np.log(5)
    )

    reset_em = getattr(
        cfg,
        "sar_reset_em",
        0.2
    )

    grad_clip = getattr(
        cfg,
        "sar_grad_clip",
        1.0
    )

    # =====================================================
    # CONFIGURE
    # =====================================================
    model = configure_model(model)

    params, _ = collect_params(

        model,

        mode=mode
    )

    optimizer = SAM(

        params,

        torch.optim.Adam,

        rho=rho,

        lr=lr
    )

    sar_model = SAR(

        model=model,

        optimizer=optimizer,

        margin_e0=margin_e0,

        reset_em=reset_em,

        steps=steps,

        episodic=True,

        grad_clip=grad_clip,
    )

    # =====================================================
    # INFERENCE
    # =====================================================
    preds_all = []

    probs_all = []

    labels_all = []

    device = next(
        model.parameters()
    ).device

    for images, labels in loader:

        images = images.to(

            device,

            non_blocking=True
        )

        logits = sar_model(images)

        probs = F.softmax(
            logits,
            dim=1
        )

        preds = probs.argmax(dim=1)

        preds_all.extend(
            preds.detach().cpu().numpy()
        )

        probs_all.append(
            probs.detach().cpu().numpy()
        )

        labels_all.extend(
            labels.numpy()
        )

    preds_all = np.array(preds_all)

    labels_all = np.array(labels_all)

    probs_all = np.concatenate(
        probs_all,
        axis=0
    )

    if return_probs:

        return (

            preds_all,

            labels_all,

            probs_all
        )

    return preds_all, labels_all

Writing /kaggle/working/tta_sar.py


In [23]:
%%writefile /kaggle/working/Final_Run.py

import os
import gc
import random
import warnings
import argparse

import torch
import numpy as np
import pandas as pd

from config import CFG
from model import get_model

warnings.filterwarnings("ignore")


# =========================================================
# DATASETS
# =========================================================
from dataset_unified import (

    get_fgadr_loader,

    get_messidor_loader,

    get_idrid_loader,

    get_deepdrid_loader,

    get_ddr_loader,
)


# =========================================================
# METHODS
# =========================================================
from baseline import run_baseline

from tta_node import run_tta as run_cfnode


# =========================================================
# SAFE TENT IMPORT
# =========================================================
try:
    from tta_tent import run_tent
except:
    from tta_tent import run_tta as run_tent


from tta_sar import run_sar

from tta_eata import run_eata

from tta_t3a import run_t3a


# =========================================================
# METRICS
# =========================================================
from metrics import compute_metrics_from_preds


# =========================================================
# AVAILABLE SEEDS
# =========================================================
AVAILABLE_SEEDS = [42, 123, 999, 2026, 777]


# =========================================================
# GLOBAL SEED
# =========================================================
def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


# =========================================================
# CLEANUP
# =========================================================
def cleanup():

    gc.collect()

    torch.cuda.empty_cache()


# =========================================================
# LOAD MODEL
# =========================================================
def load_model(cfg):

    model = get_model(cfg).to(cfg.device)

    checkpoint = torch.load(

        cfg.save_path,

        map_location=cfg.device,

        weights_only=False
    )

    if isinstance(checkpoint, dict):

        if "model" in checkpoint:

            state_dict = checkpoint["model"]

        else:

            state_dict = checkpoint

    else:

        state_dict = checkpoint

    model.load_state_dict(

        state_dict,

        strict=False
    )

    print(
        f"✅ Loaded checkpoint: {cfg.save_path}"
    )

    gpu_count = torch.cuda.device_count()

    if gpu_count > 1:

        print(
            f"🚀 Using {gpu_count} GPUs"
        )

        model = torch.nn.DataParallel(model)

    model.eval()

    return model


# =========================================================
# FRESH MODEL
# =========================================================
def get_fresh_model(cfg):

    cleanup()

    model = load_model(cfg)

    model.eval()

    return model


# =========================================================
# EVALUATE
# =========================================================
def evaluate_results(

    preds,
    labels,
    probs=None,
    ranking_scores=None
):

    return compute_metrics_from_preds(

        preds=preds,

        labels=labels,

        probs=probs,

        ranking_scores=ranking_scores
    )


# =========================================================
# PRINT METRICS
# =========================================================
def print_metrics(method_name, metrics):

    auc = metrics["auc"]

    f1 = metrics["f1"]

    acc = metrics["accuracy"]

    qwk = metrics["qwk"]

    print(f"\n{method_name}")

    print(

        f"AUC: {auc:.4f} | "

        f"F1: {f1:.4f} | "

        f"Acc: {acc:.4f} | "

        f"QWK: {qwk:.4f}"
    )


# =========================================================
# BASELINE
# =========================================================
def run_baseline_method(loader, cfg):

    print("🔥 BASELINE")

    model = get_fresh_model(cfg)

    preds, labels, probs = run_baseline(

        model,
        loader,
        return_probs=True
    )

    metrics = evaluate_results(

        preds,
        labels,
        probs=probs
    )

    del model
    cleanup()

    return metrics


# =========================================================
# TENT
# =========================================================
def run_tent_method(loader, cfg):

    print("🔥 TENT")

    model = get_fresh_model(cfg)

    preds, labels, probs = run_tent(

        model,
        loader,
        cfg,
        return_probs=True
    )

    metrics = evaluate_results(

        preds,
        labels,
        probs=probs
    )

    del model
    cleanup()

    return metrics


# =========================================================
# SAR
# =========================================================
def run_sar_method(loader, cfg):

    print("🔥 SAR")

    model = get_fresh_model(cfg)

    preds, labels, probs = run_sar(

        model,
        loader,
        cfg,
        return_probs=True
    )

    metrics = evaluate_results(

        preds,
        labels,
        probs=probs
    )

    del model
    cleanup()

    return metrics


# =========================================================
# EATA
# =========================================================
def run_eata_method(loader, cfg):

    print("🔥 EATA")

    model = get_fresh_model(cfg)

    preds, labels, probs = run_eata(

        model,
        loader,
        cfg,
        fisher_loader=None,
        return_probs=True
    )

    metrics = evaluate_results(

        preds,
        labels,
        probs=probs
    )

    del model
    cleanup()

    return metrics


# =========================================================
# T3A
# =========================================================
def run_t3a_method(loader, cfg):

    print("🔥 T3A")

    model = get_fresh_model(cfg)

    preds, labels, probs = run_t3a(

        model,
        loader,
        cfg,
        return_probs=True
    )

    metrics = evaluate_results(

        preds,
        labels,
        probs=probs
    )

    del model
    cleanup()

    return metrics


# =========================================================
# CF-NODE
# =========================================================
def run_cfnode_method(loader, cfg):

    print("🔥 CF-NODE")

    model = get_fresh_model(cfg)

    preds, labels, probs, ranking_scores = run_cfnode(

        model,
        loader,
        cfg,
        return_probs=True
    )

    metrics = evaluate_results(

        preds,
        labels,
        probs=probs,
        ranking_scores=ranking_scores
    )

    del model
    cleanup()

    return metrics


# =========================================================
# RUN ALL METHODS
# =========================================================
def run_all_methods(loader, cfg):

    all_results = {}

    all_results["BASELINE"] = run_baseline_method(
        loader,
        cfg
    )

    all_results["TENT"] = run_tent_method(
        loader,
        cfg
    )

    all_results["SAR"] = run_sar_method(
        loader,
        cfg
    )

    all_results["EATA"] = run_eata_method(
        loader,
        cfg
    )

    all_results["T3A"] = run_t3a_method(
        loader,
        cfg
    )

    all_results["CF-NODE"] = run_cfnode_method(
        loader,
        cfg
    )

    return all_results


# =========================================================
# BUILD DATASETS
# =========================================================
def build_datasets(cfg):

    datasets = {

        "FGADR": get_fgadr_loader(cfg),

        "MESSIDOR": get_messidor_loader(cfg),

        "IDRID": get_idrid_loader(cfg),

        "DeepDRiD": get_deepdrid_loader(cfg),

        "DDR": get_ddr_loader(cfg),
    }

    return datasets


# =========================================================
# MAIN
# =========================================================
def main():

    # =====================================================
    # ARGUMENTS
    # =====================================================
    parser = argparse.ArgumentParser()

    parser.add_argument(

        "--seed",

        type=int,

        default=42,

        help="Single seed to run"
    )

    args = parser.parse_args()

    seed = args.seed

    print("\n")
    print("#" * 70)
    print(f"#################### SEED = {seed} ####################")
    print("#" * 70)

    # =====================================================
    # CFG
    # =====================================================
    cfg = CFG()

    cfg.seed = seed

    cfg.num_classes = 5

    set_seed(seed)

    # =====================================================
    # TENT
    # =====================================================
    cfg.tent_mode = "classifier_only"

    cfg.tent_lr = 1e-4

    cfg.tent_steps = 1

    cfg.tent_episodic = True

    cfg.tent_grad_clip = 1.0

    # =====================================================
    # SAR
    # =====================================================
    cfg.sar_mode = "classifier_only"

    cfg.sar_lr = 1e-4

    cfg.sar_steps = 1

    cfg.sar_rho = 0.05

    cfg.sar_margin_e0 = 0.4 * np.log(cfg.num_classes)

    cfg.sar_reset_em = 0.2

    cfg.sar_grad_clip = 1.0

    # =====================================================
    # EATA
    # =====================================================
    cfg.eata_mode = "classifier_only"

    cfg.eata_lr = 1e-4

    cfg.eata_steps = 1

    cfg.eata_margin = 1.4

    cfg.eata_e_margin = 0.4 * np.log(cfg.num_classes)

    cfg.eata_d_margin = 0.05

    cfg.eata_use_fisher = False

    cfg.eata_fisher_alpha = 2000.0

    cfg.eata_fisher_steps = 200

    cfg.eata_grad_clip = 1.0

    # =====================================================
    # T3A
    # =====================================================
    cfg.t3a_max_supports = 20

    # =====================================================
    # STORAGE
    # =====================================================
    all_rows = []

    # =====================================================
    # DATASETS
    # =====================================================
    datasets = build_datasets(cfg)

    # =====================================================
    # RUN DATASETS
    # =====================================================
    for dataset_name, loader in datasets.items():

        cfg.target_dataset = dataset_name.lower()

        print("\n")
        print("=" * 60)

        print(
            f"DATASET: EyePACS → {dataset_name}"
        )

        print("=" * 60)

        results = run_all_methods(

            loader,
            cfg
        )

        print("\n########## RESULTS ##########")

        for method_name, metrics in results.items():

            print_metrics(
                method_name,
                metrics
            )

            all_rows.append({

                "Seed": seed,

                "Dataset": dataset_name,

                "Method": method_name,

                "AUC": metrics["auc"],

                "F1": metrics["f1"],

                "Accuracy": metrics["accuracy"],

                "QWK": metrics["qwk"]
            })

        cleanup()

    # =====================================================
    # SAVE CSV
    # =====================================================
    df = pd.DataFrame(all_rows)

    save_path = f"/kaggle/working/results_seed_{seed}.csv"

    df.to_csv(

        save_path,

        index=False
    )

    print("\n")
    print("=" * 70)

    print("✅ FINISHED")

    print(f"✅ Saved CSV: {save_path}")

    print("=" * 70)


# =========================================================
# ENTRY
# =========================================================
if __name__ == "__main__":

    main()

Overwriting /kaggle/working/Final_Run.py


In [24]:
!python /kaggle/working/Final_Run.py --seed 2026



######################################################################
#################### SEED = 2026 ####################
######################################################################

✅ FGADR samples: 1842

FGADR Label Distribution:
label
0    101
1    212
2    595
3    647
4    287
Name: count, dtype: int64

FGADR Example Images:
0    0000_1.png
1    0001_2.png
2    0002_1.png
3    0003_3.png
4    0004_1.png
Name: image, dtype: object


DATASET: EyePACS → FGADR
🔥 BASELINE
✅ Loaded checkpoint: /kaggle/input/models/knightcracker/seed2026-model/pytorch/default/1/best_dsr50_balanced_seed_2026.pth
🚀 Using 2 GPUs
🔥 TENT
✅ Loaded checkpoint: /kaggle/input/models/knightcracker/seed2026-model/pytorch/default/1/best_dsr50_balanced_seed_2026.pth
🚀 Using 2 GPUs
🔥 SAR
✅ Loaded checkpoint: /kaggle/input/models/knightcracker/seed2026-model/pytorch/default/1/best_dsr50_balanced_seed_2026.pth
🚀 Using 2 GPUs
🔥 EATA
✅ Loaded checkpoint: /kaggle/input/models/knightcracker/seed2026-model/py